# Transfer-Learning, Multi-Task Applicability-Domain Pipeline for PSMA Inhibitors

This notebook predicts **pIC50** and **pKi** for PSMA (glutamate carboxypeptidase II /
FOLH1) inhibitors and screens a compound library, using **transfer learning**:

1. **Pretrain** a shared neural-network trunk on a **broad ChEMBL bioactivity set**
   spanning many targets, learning a general-purpose structure→potency representation.
2. **Fine-tune** that trunk (with two task-specific heads) on a **PSMA-specific,
   multi-task BindingDB dataset** (pIC50 *and* pKi jointly, with masked loss for
   compounds that only have one of the two endpoints).
3. Compare the fine-tuned network against a tuned **LightGBM baseline** trained
   directly on the BindingDB data, blend the two, and predict on a **screening
   library**, reporting **Applicability Domain (AD)** flags and **prediction
   uncertainty** alongside every prediction.

Organized in five parts:

- **Part A — Data acquisition**: broad ChEMBL pretraining set (`chembl_webresource_client`),
  PSMA-specific BindingDB fine-tuning set (DuckDB SQL over the full TSV dump, pulling
  **both** IC50 and Ki), and the curated screening library.
- **Part B — Featurization & Applicability Domain**: the same RDKit standardization /
  fingerprint / descriptor wrapper and AD metrics (Tanimoto, descriptor-space and
  latent-space Manhattan distance, k-NN unfamiliarity) as before, calibrated on the
  PSMA-specific (fine-tuning) training set.
- **Part C — Transfer-learning multi-task model**: the pretrain → fine-tune neural
  network, plus a tuned LightGBM baseline per task.
- **Part D — Hyperparameter tuning**: `GridSearchCV`, Bayesian search via
  `BayesSearchCV` (scikit-optimize), and Bayesian/TPE search via `Optuna`, compared
  head-to-head for both model families.
- **Part E — Prediction, uncertainty, export**: predictions + bootstrap/MC-dropout
  uncertainty + AD flags for the screening library, saved to disk.

Runs anywhere with internet access (Colab local runtime, a plain VM, etc.) — nothing
depends on Colab-only services. Every network call has a **synthetic-data fallback**
so the whole notebook still runs end-to-end (with clearly-labeled fake data) if
ChEMBL/BindingDB are briefly unreachable, which is useful for dry-running the pipeline
or for CI-style smoke tests.

Everything project-specific lives in the single `CONFIG` object in **Section 1**.


## 0. Setup

Install dependencies. Safe to re-run — pip skips anything already satisfied.


In [ ]:
%pip install -q duckdb rdkit pandas numpy scikit-learn matplotlib seaborn requests tqdm scipy joblib \
    lightgbm optuna scikit-optimize torch chembl_webresource_client


In [ ]:
import os
import io
import re
import json
import zipfile
import shutil
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Sequence, List, Dict, Tuple

import duckdb
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, MACCSkeys, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import DataStructs

from sklearn.model_selection import train_test_split, KFold, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from sklearn.cross_decomposition import PLSRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import make_scorer
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib

import torch
import torch.nn as nn

import optuna
from skopt import BayesSearchCV
from skopt.space import Integer, Real

from lightgbm import LGBMRegressor

import matplotlib.pyplot as plt
import seaborn as sns

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid", context="notebook")

np.random.seed(0)
torch.manual_seed(0)

# ---- GPU support -----------------------------------------------------------
# Every NN training/inference function below (`pretrain_trunk`, `finetune_multitask`,
# `predict_with_mc_dropout`) accepts this `DEVICE` and moves its model/tensors onto it,
# so the entire pretrain -> fine-tune -> MC-dropout pipeline runs on GPU automatically
# when one is available (CUDA, or Apple Silicon MPS), and falls back to CPU otherwise.
# On Colab: Runtime -> Change runtime type -> select a GPU, then re-run from the top.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"GPU detected: {torch.cuda.get_device_name(0)} -> using CUDA.")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Apple Silicon GPU detected -> using MPS.")
else:
    DEVICE = torch.device("cpu")
    print("No GPU detected -> using CPU. (Colab: Runtime > Change runtime type > GPU.)")


def rmse(y_true, y_pred) -> float:
    '''Root-mean-squared-error, computed manually so this works across sklearn versions
    (newer sklearn removed mean_squared_error's `squared=False` argument).'''
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


print("Environment ready.")


## 1. Configuration

Everything that changes between projects lives here: ChEMBL pretraining scope, the
BindingDB target/endpoints, screening library path, AD thresholds, model
architectures, and every hyperparameter-search setting. Nothing below this cell
needs to change for a routine re-run.


In [ ]:
@dataclass
class Config:
    # ---- ChEMBL pretraining set (domain-targeted by default) -------------------
    # A bioactivity set used only to *pretrain* the shared NN trunk (transfer
    # learning) before it ever sees PSMA data. Pulled live via
    # chembl_webresource_client; cached to `chembl_pretrain_cache` so re-runs are free.
    #
    # "domain_targeted" (default): restrict pretraining to targets related to PSMA —
    # PSMA (GCPII/FOLH1) is a zinc-dependent, co-catalytic-dinuclear-zinc metallopeptidase
    # (MEROPS clan MH, family M28). Other zinc metallopeptidases/metalloproteases are
    # the natural "related" pool: their inhibitors are built around the same handful of
    # zinc-binding chemotypes PSMA inhibitors use (phosphonates, thiols, hydroxamates,
    # ureas), so pretraining here teaches the trunk that specific structure-activity
    # vocabulary rather than generic, unrelated bioactivity patterns.
    # "broad": the original target-agnostic sample across arbitrary human targets.
    chembl_pretrain_mode: str = "domain_targeted"     # "domain_targeted" | "broad"
    chembl_pretrain_target_keywords: Sequence[str] = field(default_factory=lambda: [
        "glutamate carboxypeptidase",       # PSMA's own family (paralogs, e.g. NAALADase L)
        "aminopeptidase",                   # other M-clan zinc peptidases, same catalytic chemistry
        "leucine aminopeptidase",
        "carboxypeptidase",
        "angiotensin-converting enzyme",    # zinc metallopeptidase; classic phosphonate/thiol ZBGs
        "neprilysin",                       # zinc metallopeptidase (NEP), thiol/phosphonate inhibitors
        "endothelin-converting enzyme",     # related M13 zinc metallopeptidase
        "thermolysin",                      # archetypal zinc metallopeptidase used to design ZBGs
        "matrix metalloproteinase",         # zinc metalloproteinases, hydroxamate ZBGs
        "insulin-degrading enzyme",         # zinc metallopeptidase
        "meprin",                           # zinc metalloproteinase
        "carbonic anhydrase",               # different fold, but a classic zinc metalloenzyme drug
                                             # target with a rich, structurally-diverse zinc-binding
                                             # inhibitor literature (sulfonamides, etc.)
    ])
    chembl_pretrain_target_type: str = "SINGLE PROTEIN"
    chembl_pretrain_min_targets_or_fallback_to_broad: int = 5   # if fewer targets match, fall back to "broad"
    chembl_pretrain_cache: str = "./ad_pipeline_data/chembl_pretrain_domain.csv"
    chembl_pretrain_standard_types: Sequence[str] = field(
        default_factory=lambda: ["IC50", "Ki", "Kd", "EC50"]
    )
    chembl_pretrain_organism: str = "Homo sapiens"
    chembl_pretrain_max_targets: int = 150      # cap on distinct targets sampled ("broad" mode only)
    chembl_pretrain_per_target_cap: int = 300   # max activities pulled per target
    chembl_pretrain_max_compounds: int = 10000   # hard cap on the final pretraining set
    chembl_pretrain_endpoint_name: str = "pX_generic"   # generic potency label used only for pretraining
    force_rebuild_chembl_cache: bool = False

    # ---- BindingDB acquisition (PSMA fine-tuning set) --------------------------
    bindingdb_tsv_zip_name: str = "BindingDB_All_202609_tsv.zip"
    bindingdb_download_url: str = (
        "https://www.bindingdb.org/rwd/bind/downloads/BindingDB_All_202609_tsv.zip"
    )
    auto_resolve_bindingdb_url: bool = True
    data_dir: str = "./ad_pipeline_data"
    force_redownload: bool = False

    # ---- Target / multi-task endpoint selection (BindingDB SQL filter) --------
    target_uniprot_id: Optional[str] = "Q04609"          # PSMA / FOLH1 / GCPII
    target_name_like: Optional[str] = "Glutamate carboxypeptidase 2"
    organism_like: Optional[str] = "Homo sapiens"

    # Multi-task: model BOTH endpoints jointly instead of picking one.
    affinity_columns: Sequence[str] = field(default_factory=lambda: ["IC50 (nM)", "Ki (nM)"])
    endpoint_names: Sequence[str] = field(default_factory=lambda: ["pIC50", "pKi"])

    max_affinity_nm: float = 1e6
    min_affinity_nm: float = 1e-3

    # ---- Curated screening library ----------------------------------------------
    screening_library_path: str = "/Enamine_screen_library.sdf"
    screening_smiles_col: str = "SMILES"
    screening_id_col: Optional[str] = "compound_id"

    # ---- Featurization ------------------------------------------------------------
    morgan_radius: int = 2
    morgan_n_bits: int = 2048
    feature_set: str = "fp_plus_descriptors"   # "fp_plus_descriptors" | "descriptors_only" | "fp_only"

    # ---- Applicability domain thresholds (calibrated on the PSMA fine-tuning set) --
    ad_percentile_threshold: float = 95.0
    unfamiliarity_k: int = 5
    latent_pca_components: int = 20

    # ---- Neural-network transfer-learning model ------------------------------------
    nn_hidden_dims: Sequence[int] = field(default_factory=lambda: (256, 128))
    nn_dropout: float = 0.2
    nn_lr: float = 1e-3
    nn_finetune_lr: float = 5e-4
    nn_weight_decay: float = 1e-5
    nn_pretrain_epochs: int = 40
    nn_finetune_total_epochs: int = 80
    nn_freeze_epochs: int = 15          # epochs of head-only training before unfreezing the trunk
    nn_batch_size: int = 32
    nn_pretrain_batch_size: int = 128
    nn_mc_dropout_passes: int = 30      # forward passes for MC-dropout uncertainty

    # ---- LightGBM baseline (trained directly on BindingDB, one model per task) -----
    n_bootstrap: int = 30               # bootstrap ensemble size for GBM uncertainty

    # ---- Hyperparameter search ------------------------------------------------------
    run_gridsearch: bool = True
    run_bayes_search: bool = True       # scikit-optimize BayesSearchCV
    run_optuna: bool = True             # Optuna TPE search for the NN
    bayes_search_n_iter: int = 20
    optuna_n_trials: int = 25
    optuna_cv_folds: int = 3            # cheap inner CV used inside the Optuna objective

    test_size: float = 0.2
    cv_folds: int = 10
    random_state: int = 42
    n_eval_repeats: int = 10          # repeated random splits for robust held-out statistics
    n_calibration_bins: int = 5       # uncertainty-calibration reliability diagram bins

    # ---- Output ----------------------------------------------------------------------
    output_dir: str = "./ad_pipeline_outputs"

    def __post_init__(self):
        Path(self.data_dir).mkdir(parents=True, exist_ok=True)
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)


CONFIG = Config()
CONFIG

## 2. Part A.1 — ChEMBL pretraining set: PSMA-related targets only

Rather than pretraining on an arbitrary cross-section of ChEMBL, this pulls
bioactivities **only from targets related to PSMA** — controlled by
`CONFIG.chembl_pretrain_target_keywords`. PSMA (GCPII/FOLH1) is a zinc-dependent,
co-catalytic-dinuclear-zinc **metallopeptidase** (MEROPS family M28), and the great
majority of known PSMA inhibitors are built around a handful of zinc-binding groups
(phosphonates, thiols, hydroxamates, ureas) shared with other zinc metallopeptidase
inhibitors. So the "related" pool queried here is: PSMA's own paralog family
(other glutamate carboxypeptidases), other aminopeptidases/carboxypeptidases,
and the classic zinc-metalloprotease drug targets whose inhibitor chemistry PSMA
inhibitors borrow from directly — ACE, neprilysin, ECE-1, thermolysin, MMPs,
insulin-degrading enzyme, meprin — plus carbonic anhydrase, included for its
structurally diverse zinc-binding inhibitor literature even though its fold differs.

This should transfer better than a target-agnostic broad set because the trunk
learns the specific structure-activity vocabulary of zinc-binding-group chemistry,
not generic bioactivity patterns from unrelated target classes. Cached to CSV so
repeated runs don't re-hit the API; if ChEMBL is unreachable, or too few related
targets are found, it falls back first to the old **broad** (target-agnostic) mode
and then to a small synthetic set, so the rest of the notebook always still runs.


In [ ]:
def _synthetic_chembl_pretrain_set(cfg: Config, n: int = 500) -> pd.DataFrame:
    '''Small, clearly-labeled synthetic stand-in used only if ChEMBL can't be reached.'''
    print("!! Using SYNTHETIC ChEMBL pretraining data (network/API unavailable) !!")
    demo_smiles = [
        "CCOc1ccc2nc(S(N)(=O)=O)sc2c1", "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
        "O=C(O)c1ccccc1O", "CC(=O)Oc1ccccc1C(=O)O",
        "Clc1ccc(C(c2ccccc2Cl)N2CCN(CCO)CC2)cc1",
        "COc1cc2c(cc1OC)C(=O)C(CC1CCN(C)CC1)C2",
        "CN1CCC[C@H]1c1cccnc1", "CC(C)NCC(O)COc1cccc2ccccc12",
        "Cc1ccc(cc1)S(=O)(=O)N", "O=C(Nc1ccccc1)c1ccccc1O",
        "CCN(CC)CCOC(=O)c1ccc(N)cc1", "COc1ccc2[nH]c(nc2c1)S(=O)Cc1ncc(C)c(OC)c1C",
    ]
    rng = np.random.RandomState(cfg.random_state)
    rows = []
    for i in range(n):
        base = demo_smiles[i % len(demo_smiles)]
        rows.append({
            "smiles": base,
            "target_chembl_id": f"SYNTH_TARGET_{i % 40}",
            cfg.chembl_pretrain_endpoint_name: float(np.clip(rng.normal(6.5, 1.5), 3.0, 10.0)),
        })
    return pd.DataFrame(rows)


def find_related_chembl_targets(cfg: Config) -> pd.DataFrame:
    '''Searches ChEMBL's target registry for every target whose preferred name matches
    one of `chembl_pretrain_target_keywords`, returning a deduplicated
    (target_chembl_id, pref_name, organism, matched_keyword) table.'''
    from chembl_webresource_client.new_client import new_client

    target_client = new_client.target
    rows = []
    for kw in cfg.chembl_pretrain_target_keywords:
        try:
            hits = target_client.filter(pref_name__icontains=kw, target_type=cfg.chembl_pretrain_target_type) \
                                 .only(["target_chembl_id", "pref_name", "organism"])
            for h in hits:
                rows.append({"target_chembl_id": h.get("target_chembl_id"), "pref_name": h.get("pref_name"),
                             "organism": h.get("organism"), "matched_keyword": kw})
        except Exception as e:
            print(f"  target search failed for keyword '{kw}': {e}")
    if not rows:
        return pd.DataFrame(columns=["target_chembl_id", "pref_name", "organism", "matched_keyword"])
    df = pd.DataFrame(rows).dropna(subset=["target_chembl_id"])
    df = df.drop_duplicates(subset=["target_chembl_id"]).reset_index(drop=True)
    return df


def fetch_activities_for_targets(cfg: Config, target_ids: Sequence[str]) -> pd.DataFrame:
    '''Pulls bioactivities restricted to a specific list of target_chembl_ids, in
    batches (long __in lists are chunked to keep each request well-formed).'''
    from chembl_webresource_client.new_client import new_client

    activity_client = new_client.activity
    rows = []
    per_target_counts: Dict[str, int] = {}
    batch_size = 40
    target_ids = list(target_ids)

    for i in range(0, len(target_ids), batch_size):
        batch = target_ids[i:i + batch_size]
        query = activity_client.filter(
            target_chembl_id__in=batch,
            standard_type__in=list(cfg.chembl_pretrain_standard_types),
            pchembl_value__isnull=False,
        ).only(["canonical_smiles", "target_chembl_id", "pchembl_value"])

        for rec in tqdm(query, desc=f"ChEMBL activities (targets {i+1}-{i+len(batch)}/{len(target_ids)})"):
            smi = rec.get("canonical_smiles")
            tgt = rec.get("target_chembl_id")
            pval = rec.get("pchembl_value")
            if not smi or not tgt or pval is None:
                continue
            if per_target_counts.get(tgt, 0) >= cfg.chembl_pretrain_per_target_cap:
                continue
            per_target_counts[tgt] = per_target_counts.get(tgt, 0) + 1
            rows.append({"smiles": smi, "target_chembl_id": tgt, cfg.chembl_pretrain_endpoint_name: float(pval)})
            if len(rows) >= cfg.chembl_pretrain_max_compounds:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)


def fetch_chembl_pretraining_set_broad(cfg: Config) -> pd.DataFrame:
    '''Original target-agnostic strategy: sample activities across arbitrary human
    targets. Used as the "broad" mode, and as an automatic fallback if domain-targeted
    search doesn't find enough related targets.'''
    from chembl_webresource_client.new_client import new_client

    activity_client = new_client.activity
    rows, seen_targets, per_target_counts = [], set(), {}
    query = activity_client.filter(
        standard_type__in=list(cfg.chembl_pretrain_standard_types),
        target_organism=cfg.chembl_pretrain_organism,
        pchembl_value__isnull=False,
    ).only(["canonical_smiles", "target_chembl_id", "pchembl_value"])

    for rec in tqdm(query, total=cfg.chembl_pretrain_max_compounds * 3, desc="ChEMBL pretraining set (broad)"):
        smi, tgt, pval = rec.get("canonical_smiles"), rec.get("target_chembl_id"), rec.get("pchembl_value")
        if not smi or not tgt or pval is None:
            continue
        if tgt not in per_target_counts and len(seen_targets) >= cfg.chembl_pretrain_max_targets:
            continue
        if per_target_counts.get(tgt, 0) >= 60:
            continue
        seen_targets.add(tgt)
        per_target_counts[tgt] = per_target_counts.get(tgt, 0) + 1
        rows.append({"smiles": smi, "target_chembl_id": tgt, cfg.chembl_pretrain_endpoint_name: float(pval)})
        if len(rows) >= cfg.chembl_pretrain_max_compounds:
            break
    return pd.DataFrame(rows)


def fetch_chembl_pretraining_set(cfg: Config) -> pd.DataFrame:
    cache_path = Path(cfg.chembl_pretrain_cache)
    if cache_path.exists() and not cfg.force_rebuild_chembl_cache:
        print(f"Found cached ChEMBL pretraining set: {cache_path}")
        return pd.read_csv(cache_path)

    try:
        if cfg.chembl_pretrain_mode == "domain_targeted":
            print(f"Searching ChEMBL for targets related to PSMA via {len(cfg.chembl_pretrain_target_keywords)} "
                  f"keyword(s): {list(cfg.chembl_pretrain_target_keywords)}")
            targets_df = find_related_chembl_targets(cfg)
            print(f"Found {len(targets_df):,} matching targets.")
            if len(targets_df):
                print(targets_df[["pref_name", "organism", "matched_keyword"]].to_string(index=False))

            if len(targets_df) < cfg.chembl_pretrain_min_targets_or_fallback_to_broad:
                print(f"Fewer than {cfg.chembl_pretrain_min_targets_or_fallback_to_broad} related targets found "
                      f"-> falling back to 'broad' mode for this run.")
                df = fetch_chembl_pretraining_set_broad(cfg)
            else:
                df = fetch_activities_for_targets(cfg, targets_df["target_chembl_id"].tolist())
                df = df.merge(targets_df[["target_chembl_id", "pref_name", "matched_keyword"]],
                               on="target_chembl_id", how="left")
        else:
            df = fetch_chembl_pretraining_set_broad(cfg)

        if not len(df):
            raise RuntimeError("ChEMBL query returned no usable rows.")

        df = df.drop_duplicates(subset=["smiles"]).reset_index(drop=True)
        print(f"\nFetched {len(df):,} ChEMBL pretraining compounds across "
              f"{df['target_chembl_id'].nunique():,} targets.")
        if "pref_name" in df.columns:
            print("Compounds per target:")
            print(df["pref_name"].value_counts().head(20).to_string())

    except Exception as e:
        print(f"Could not fetch live ChEMBL data ({e}); falling back to a synthetic set.")
        df = _synthetic_chembl_pretrain_set(cfg)

    df.to_csv(cache_path, index=False)
    print(f"Cached ChEMBL pretraining set to {cache_path}")
    return df


pretrain_df = fetch_chembl_pretraining_set(CONFIG)
pretrain_df.head()


## 3. Part A.2 — BindingDB PSMA fine-tuning set (multi-task: pIC50 + pKi)

Same DuckDB-over-the-full-TSV approach as before, but now pulling **both** affinity
columns in `CONFIG.affinity_columns` (IC50 and Ki) and pivoting to a **wide,
multi-task table**: one row per compound with a `pIC50` column, a `pKi` column, and
`NaN` in whichever one a given compound doesn't have a measurement for. Downstream,
the multi-task model trains on both columns at once with a masked loss, so compounds
don't need *both* labels to contribute.


In [ ]:
def resolve_bindingdb_tsv_url(cfg: Config) -> tuple:
    '''Scrape the BindingDB download page for the CURRENT 'BindingDB_All_<YYYYMM>_tsv.zip'
    filename (BindingDB republishes this monthly, so a hardcoded name goes stale).'''
    download_page = "https://www.bindingdb.org/rwd/bind/chemsearch/marvin/Download.jsp"
    try:
        resp = requests.get(download_page, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        resp.raise_for_status()
        m = re.search(r"download_file=(/rwd/bind/downloads/BindingDB_All_\d{6}_tsv\.zip)", resp.text)
        if m:
            file_param = m.group(1)
            zip_name = file_param.rsplit("/", 1)[-1]
            direct_url = f"https://www.bindingdb.org{file_param}"
            print(f"Resolved current BindingDB TSV: {zip_name}")
            return direct_url, zip_name
        print(f"Could not find the current filename on {download_page}; using CONFIG default.")
    except Exception as e:
        print(f"Could not reach {download_page} ({e}); using CONFIG default.")
    return cfg.bindingdb_download_url, cfg.bindingdb_tsv_zip_name


def _fetch_to_file(url: str, dest: Path) -> str:
    headers = {"User-Agent": "Mozilla/5.0 (compatible; AD-pipeline/1.0)"}
    with requests.get(url, stream=True, timeout=120, headers=headers) as r:
        r.raise_for_status()
        content_type = r.headers.get("content-type", "")
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as pbar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                pbar.update(len(chunk))
    return content_type


def _recover_real_download_link(html_path: Path) -> Optional[str]:
    try:
        text = html_path.read_text(errors="ignore")
    except Exception:
        return None
    m = re.search(r'(https://www\.bindingdb\.org/rwd/bind/downloads/[^\s"\x27<>)]+\.zip)', text)
    return m.group(1) if m else None


def download_bindingdb(cfg: Config) -> Optional[Path]:
    '''Download (if needed) and unzip the full BindingDB TSV dump. Returns None (instead of
    raising) if the download can't be completed, so the notebook can fall back to a
    synthetic fine-tuning set and still run end-to-end.'''
    data_dir = Path(cfg.data_dir)
    try:
        if cfg.auto_resolve_bindingdb_url:
            url, zip_name = resolve_bindingdb_tsv_url(cfg)
        else:
            url, zip_name = cfg.bindingdb_download_url, cfg.bindingdb_tsv_zip_name

        zip_path = data_dir / zip_name
        tsv_path = data_dir / zip_name.replace(".zip", "")

        if tsv_path.exists() and not cfg.force_redownload:
            print(f"Found existing TSV: {tsv_path}")
            return tsv_path

        if not zip_path.exists() or cfg.force_redownload:
            print(f"Downloading {url} ...")
            content_type = _fetch_to_file(url, zip_path)
            print(f"Saved to {zip_path} (content-type: {content_type})")

            if not zipfile.is_zipfile(zip_path):
                real_url = _recover_real_download_link(zip_path)
                if real_url and real_url != url:
                    print(f"Not a zip — retrying with the real link found on the landing page: {real_url}")
                    _fetch_to_file(real_url, zip_path)

        if not zipfile.is_zipfile(zip_path):
            with open(zip_path, "rb") as f:
                head = f.read(300)
            zip_path.unlink(missing_ok=True)
            raise RuntimeError(f"Downloaded file is not a valid zip. First bytes: {head!r}")

        print("Unzipping ...")
        with zipfile.ZipFile(zip_path) as zf:
            member = [m for m in zf.namelist() if m.lower().endswith(".tsv")][0]
            with zf.open(member) as src, open(tsv_path, "wb") as dst:
                shutil.copyfileobj(src, dst)
        print(f"Extracted: {tsv_path}")
        return tsv_path

    except Exception as e:
        print(f"Could not obtain the BindingDB TSV dump ({e}).")
        return None


def query_bindingdb_duckdb_multitask(tsv_path: Path, cfg: Config) -> pd.DataFrame:
    '''Pull one long dataframe with a row per (compound, endpoint) for every column in
    `cfg.affinity_columns`, using DuckDB SQL streaming over the TSV (never materialized
    fully in pandas).'''
    con = duckdb.connect(database=":memory:")
    con.execute("PRAGMA threads=4;")
    con.execute(f'''
        CREATE OR REPLACE VIEW bdb AS
        SELECT * FROM read_csv(
            '{tsv_path.as_posix()}', delim='\t', header=True, quote='"', escape='"',
            strict_mode=False, ignore_errors=True, all_varchar=True
        )
    ''')

    base_where = ['"Ligand SMILES" IS NOT NULL']
    if cfg.target_uniprot_id:
        uniprot_conditions = [
            f'"UniProt (SwissProt) Primary ID of Target Chain {i}" = \'{cfg.target_uniprot_id}\''
            for i in range(1, 6)
        ]
        base_where.append("(" + " OR ".join(uniprot_conditions) + ")")
    if cfg.target_name_like:
        safe = cfg.target_name_like.replace("'", "''")
        base_where.append(f'"Target Name" ILIKE \'%{safe}%\'')
    if cfg.organism_like:
        safe = cfg.organism_like.replace("'", "''")
        base_where.append(f'"Target Source Organism According to Curator or DataSource" ILIKE \'%{safe}%\'')

    frames = []
    for col, endpoint in zip(cfg.affinity_columns, cfg.endpoint_names):
        where_sql = " AND ".join(base_where + [f'"{col}" IS NOT NULL'])
        query = f'''
            SELECT
                "Ligand SMILES"                                    AS smiles,
                "BindingDB Ligand Name"                            AS ligand_name,
                "{col}"                                            AS affinity_raw,
                '{endpoint}'                                       AS endpoint_name
            FROM bdb
            WHERE {where_sql}
        '''
        part = con.execute(query).fetch_df()
        print(f"  {endpoint} ('{col}'): {len(part):,} raw rows.")
        frames.append(part)
    con.close()
    return pd.concat(frames, ignore_index=True)


def _synthetic_bindingdb_finetune_set(cfg: Config, n: int = 250) -> pd.DataFrame:
    print("!! Using SYNTHETIC BindingDB PSMA fine-tuning data (network/download unavailable) !!")
    demo_smiles = [
        "CCOc1ccc2nc(S(N)(=O)=O)sc2c1", "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
        "O=C(O)c1ccccc1O", "CC(=O)Oc1ccccc1C(=O)O",
        "Clc1ccc(C(c2ccccc2Cl)N2CCN(CCO)CC2)cc1",
        "COc1cc2c(cc1OC)C(=O)C(CC1CCN(C)CC1)C2",
        "CN1CCC[C@H]1c1cccnc1", "CC(C)NCC(O)COc1cccc2ccccc12",
        "NC(CCC(=O)O)C(=O)O", "OC(=O)CCC(NC(=O)NC(CCC(=O)O)C(=O)O)C(=O)O",
    ]
    rng = np.random.RandomState(cfg.random_state)
    rows = []
    for i in range(n):
        smi = demo_smiles[i % len(demo_smiles)]
        row = {"smiles": smi, "ligand_name": f"synthetic_{i}"}
        if rng.rand() < 0.75:
            row[cfg.endpoint_names[0]] = float(np.clip(rng.normal(6.8, 1.2), 3, 10))
        if rng.rand() < 0.6:
            row[cfg.endpoint_names[1]] = float(np.clip(rng.normal(6.5, 1.3), 3, 10))
        rows.append(row)
    return pd.DataFrame(rows)


def clean_bindingdb_multitask(df_long: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    '''Coerce to numeric, drop censored (>,<) measurements, compute pAffinity, take the
    per-compound median for each endpoint separately, then pivot wide so every
    compound is one row with (possibly-NaN) pIC50 / pKi columns.'''
    df = df_long.copy()
    df["is_censored"] = df["affinity_raw"].astype(str).str.contains("[<>]", regex=True)
    df = df[~df["is_censored"]]
    df["affinity_nm"] = pd.to_numeric(df["affinity_raw"], errors="coerce")
    df = df.dropna(subset=["affinity_nm", "smiles"])
    df = df[(df["affinity_nm"] >= cfg.min_affinity_nm) & (df["affinity_nm"] <= cfg.max_affinity_nm)]
    df["pAffinity"] = 9 - np.log10(df["affinity_nm"])

    per_endpoint = (
        df.groupby(["smiles", "endpoint_name"], as_index=False)
        .agg({"pAffinity": "median", "ligand_name": "first"})
    )
    wide = per_endpoint.pivot_table(index="smiles", columns="endpoint_name", values="pAffinity")
    names = per_endpoint.groupby("smiles")["ligand_name"].first()
    wide = wide.join(names).reset_index()
    for ep in cfg.endpoint_names:
        if ep not in wide.columns:
            wide[ep] = np.nan
    print(f"Cleaned multi-task fine-tuning set: {len(wide):,} unique compounds "
          + ", ".join(f"({ep}: {wide[ep].notna().sum():,})" for ep in cfg.endpoint_names))
    return wide.reset_index(drop=True)


# --- run it -------------------------------------------------------------------
tsv_path = download_bindingdb(CONFIG)
if tsv_path is not None:
    raw_bdb_df = query_bindingdb_duckdb_multitask(tsv_path, CONFIG)
    train_df = clean_bindingdb_multitask(raw_bdb_df, CONFIG)
else:
    train_df = _synthetic_bindingdb_finetune_set(CONFIG)

train_df.head()


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully inside Colab.")
except ModuleNotFoundError:
    print("Running on a local environment; skipping Google Drive mount as it is not needed.")

## 4. Part A.3 — Curated screening library

Unchanged from the original pipeline: the compounds you want predictions +
applicability-domain flags for. Supports CSV (with a SMILES column) or SDF.


In [ ]:
def load_screening_library(cfg: Config) -> pd.DataFrame:
    # Sanitize path: if it starts with an absolute root slash like '/Enamine_screen_library.sdf',
    # convert it to a local relative path './Enamine_screen_library.sdf' to avoid Windows permission errors.
    raw_path = cfg.screening_library_path
    if raw_path.startswith("/") or raw_path.startswith("\\"):
        raw_path = "." + raw_path

    path = Path(raw_path)
    if not path.exists():
        print(f"Screening library not found at {path}. Generating a demo screening library locally.")
        path.parent.mkdir(parents=True, exist_ok=True)
        demo = pd.DataFrame({
            "compound_id": [f"demo_{i}" for i in range(8)],
            "SMILES": [
                "CCOc1ccc2nc(S(N)(=O)=O)sc2c1",
                "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
                "O=C(O)c1ccccc1O",
                "CC(=O)Oc1ccccc1C(=O)O",
                "Clc1ccc(C(c2ccccc2Cl)N2CCN(CCO)CC2)cc1",
                "COc1cc2c(cc1OC)C(=O)C(CC1CCN(C)CC1)C2",
                "CN1CCC[C@H]1c1cccnc1",
                "CC(C)NCC(O)COc1cccc2ccccc12"
            ]
        })
        demo.to_csv(path, index=False)
        print(f"Demo screening library saved to: {path}")

    if path.suffix.lower() in (".sdf", ".sd"):
        suppl = Chem.SDMolSupplier(str(path))
        rows = []
        for i, mol in enumerate(suppl):
            if mol is None:
                continue
            smi = Chem.MolToSmiles(mol)
            cid = mol.GetProp("_Name") if mol.HasProp("_Name") else f"SDF_mol_{i}"
            rows.append({"smiles": smi, "compound_id": cid})
        return pd.DataFrame(rows)
    else:
        df = pd.read_csv(path)
        df = df.rename(columns={cfg.screening_smiles_col: "smiles"})
        if cfg.screening_id_col and cfg.screening_id_col in df.columns:
            df = df.rename(columns={cfg.screening_id_col: "compound_id"})
        elif "compound_id" not in df.columns:
            df["compound_id"] = [f"ROW_{i}" for i in range(len(df))]
        return df[["smiles", "compound_id"]]

screen_df = load_screening_library(CONFIG)
screen_df.head()

## 5. Part B.1 — Featurization

Same RDKit standardization + Morgan-fingerprint + descriptor wrapper as the original
pipeline, now applied to **three** sets: the broad ChEMBL pretraining set, the PSMA
BindingDB fine-tuning set, and the screening library. Using the identical featurizer
everywhere is what makes the pretrained trunk's weights meaningful once loaded into
the fine-tuning model — the input feature space has to match.


In [ ]:
class RDKitFeaturizer:
    '''Reusable RDKit wrapper: standardization + fingerprints + descriptors.

    Uses RDKit's full 2D descriptor catalog (~200 physicochemical/topological/graph
    descriptors from `Descriptors._descList`, in the spirit of AutoQSAR's ~500-descriptor
    pool[1]) rather than a small hand-picked subset. The redundancy this introduces is
    dealt with explicitly in Part B.1b below, following AutoQSAR's own near-constant +
    correlation-clustering filter, rather than left for the model to sort out.

    [1] Dixon et al., "AutoQSAR: an automated machine learning tool for best-practice
        QSAR modeling," Future Med. Chem. 2016.
    '''

    DESCRIPTOR_NAMES = [name for name, _ in Descriptors._descList]

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self._normalizer = rdMolStandardize.Normalizer()
        self._lfc = rdMolStandardize.LargestFragmentChooser()
        self._uncharger = rdMolStandardize.Uncharger()
        self._desc_fns = dict(Descriptors._descList)

    def standardize(self, smiles: str) -> Optional[Chem.Mol]:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        try:
            mol = self._normalizer.normalize(mol)
            mol = self._lfc.choose(mol)
            mol = self._uncharger.uncharge(mol)
            Chem.SanitizeMol(mol)
        except Exception:
            return None
        return mol

    def morgan_fp(self, mol: Chem.Mol):
        return AllChem.GetMorganFingerprintAsBitVect(mol, self.cfg.morgan_radius, nBits=self.cfg.morgan_n_bits)

    def morgan_fp_with_bit_info(self, mol: Chem.Mol):
        '''Same fingerprint as `morgan_fp`, but also returns RDKit's bitInfo dict
        (bit -> [(center_atom_idx, radius), ...]), needed to map bits back to atoms
        for the interpretability analysis in Part E.6.'''
        bit_info: Dict[int, list] = {}
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol, self.cfg.morgan_radius, nBits=self.cfg.morgan_n_bits, bitInfo=bit_info
        )
        return fp, bit_info

    def descriptors(self, mol: Chem.Mol) -> np.ndarray:
        vals = []
        for name in self.DESCRIPTOR_NAMES:
            fn = self._desc_fns.get(name)
            try:
                vals.append(fn(mol) if fn else np.nan)
            except Exception:
                vals.append(np.nan)
        return np.array(vals, dtype=float)

    def featurize_dataframe(self, df: pd.DataFrame, smiles_col: str = "smiles") -> dict:
        mols, fps, desc_rows, keep_idx = [], [], [], []
        for i, smi in enumerate(tqdm(df[smiles_col].tolist(), desc="Featurizing")):
            mol = self.standardize(smi)
            if mol is None:
                continue
            mols.append(mol)
            fps.append(self.morgan_fp(mol))
            desc_rows.append(self.descriptors(mol))
            keep_idx.append(i)
        desc_arr = np.vstack(desc_rows) if desc_rows else np.empty((0, len(self.DESCRIPTOR_NAMES)))
        clean_df = df.iloc[keep_idx].reset_index(drop=True)
        n_dropped = len(df) - len(clean_df)
        if n_dropped:
            print(f"  dropped {n_dropped} molecules that failed standardization/parsing.")
        return {"df": clean_df, "mols": mols, "fps": fps, "descriptors": desc_arr,
                "descriptor_names": self.DESCRIPTOR_NAMES}


featurizer = RDKitFeaturizer(CONFIG)
pretrain_feat = featurizer.featurize_dataframe(pretrain_df, smiles_col="smiles")
train_feat = featurizer.featurize_dataframe(train_df, smiles_col="smiles")
screen_feat = featurizer.featurize_dataframe(screen_df, smiles_col="smiles")

pretrain_df = pretrain_feat["df"]
train_df = train_feat["df"]
screen_df = screen_feat["df"]

pretrain_fps, pretrain_desc_raw = pretrain_feat["fps"], pretrain_feat["descriptors"]
train_fps, train_desc_raw = train_feat["fps"], train_feat["descriptors"]
screen_fps, screen_desc_raw = screen_feat["fps"], screen_feat["descriptors"]
descriptor_names_raw = pretrain_feat["descriptor_names"]

print(f"ChEMBL pretraining set after featurization: {len(pretrain_df):,}")
print(f"BindingDB PSMA fine-tuning set after featurization: {len(train_df):,}")
print(f"Screening set after featurization: {len(screen_df):,}")
print(f"Raw descriptor pool: {len(descriptor_names_raw)} descriptors (before redundancy filtering below).")


## 5b. Part B.1b — Descriptor curation (near-constant + correlation-cluster filtering)

Directly follows the AutoQSAR paper's two-stage descriptor curation, applied to the
broad ChEMBL pretraining set (the largest, most chemically diverse sample, so its
correlation structure generalizes best):

1. **Drop near-constant / near-zero-variance descriptors** — a descriptor where ≥90%
   of compounds share one value carries almost no information (e.g., "number of
   tetrazoles" is 0 for nearly everything).
2. **Cluster the survivors by |Pearson correlation|** (hierarchical, complete-linkage,
   cut at r = 0.8) and keep only the descriptor closest to each cluster's centroid.
   This directly addresses the collinearity in a ~200-descriptor pool without
   depending on any one model family to "figure it out" from redundant inputs.

The same descriptor columns (fit once, on the pretraining set) are then applied to
the fine-tuning and screening descriptor matrices, so everything downstream — the
shared `feature_scaler`, the NN trunk, the GBM/KPLS baselines — sees a consistent,
de-redundified feature space.


In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform


class AutoQSARDescriptorSelector:
    '''Near-constant filter + correlation-cluster redundancy reduction, as described in
    the AutoQSAR paper (Dixon et al., Future Med. Chem. 2016).'''

    def __init__(self, near_constant_frac: float = 0.90, corr_threshold: float = 0.8):
        self.near_constant_frac = near_constant_frac
        self.corr_threshold = corr_threshold
        self.selected_idx: Optional[np.ndarray] = None
        self.selected_names: Optional[List[str]] = None

    def fit(self, X: np.ndarray, names: Sequence[str]) -> "AutoQSARDescriptorSelector":
        X = np.nan_to_num(X)
        n = len(X)

        # Stage 1: near-constant filter.
        keep1 = []
        for j in range(X.shape[1]):
            col = X[:, j]
            _, counts = np.unique(col, return_counts=True)
            most_common_frac = counts.max() / n
            if most_common_frac < self.near_constant_frac and np.var(col) > 1e-8:
                keep1.append(j)
        if not keep1:
            keep1 = list(range(X.shape[1]))
        X1 = X[:, keep1]
        names1 = [names[j] for j in keep1]
        print(f"  Stage 1 (near-constant filter): {X.shape[1]} -> {X1.shape[1]} descriptors.")

        # Stage 2: correlation-cluster redundancy reduction.
        if X1.shape[1] > 1:
            corr = np.corrcoef(X1, rowvar=False)
            corr = np.nan_to_num(corr, nan=0.0)
            np.fill_diagonal(corr, 1.0)
            dist = 1.0 - np.abs(corr)
            dist = (dist + dist.T) / 2.0        # enforce exact symmetry (float noise)
            np.fill_diagonal(dist, 0.0)
            condensed = squareform(dist, checks=False)
            Z = linkage(condensed, method="complete")
            cluster_labels = fcluster(Z, t=1.0 - self.corr_threshold, criterion="distance")

            keep2 = []
            for cl in np.unique(cluster_labels):
                members = np.where(cluster_labels == cl)[0]
                if len(members) == 1:
                    keep2.append(members[0])
                    continue
                sub_corr = np.abs(corr[np.ix_(members, members)])
                centroid_scores = sub_corr.mean(axis=1)   # closest-to-centroid = highest mean similarity
                keep2.append(members[np.argmax(centroid_scores)])
            keep2 = sorted(keep2)
        else:
            keep2 = list(range(X1.shape[1]))

        self.selected_idx = np.array([keep1[j] for j in keep2])
        self.selected_names = [names[j] for j in self.selected_idx]
        print(f"  Stage 2 (correlation clustering, r<={self.corr_threshold}): "
              f"{X1.shape[1]} -> {len(self.selected_idx)} descriptors.")
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        return np.nan_to_num(X)[:, self.selected_idx]


print("Fitting AutoQSAR-style descriptor selector on the broad ChEMBL pretraining set...")
descriptor_selector = AutoQSARDescriptorSelector(corr_threshold=0.8).fit(pretrain_desc_raw, descriptor_names_raw)

pretrain_desc = descriptor_selector.transform(pretrain_desc_raw)
train_desc = descriptor_selector.transform(train_desc_raw)
screen_desc = descriptor_selector.transform(screen_desc_raw)
descriptor_names = descriptor_selector.selected_names

print(f"Final curated descriptor set ({len(descriptor_names)}): {descriptor_names[:12]}"
      f"{' ...' if len(descriptor_names) > 12 else ''}")


## 6. Part B.2 — Applicability Domain

Unchanged logic from the original pipeline: four complementary distance-from-training
metrics (Tanimoto dissimilarity on fingerprints, Manhattan distance in descriptor
space, Manhattan distance in a PCA "latent" proxy, and k-NN unfamiliarity), each
calibrated against a percentile of the **training set's own internal nearest-neighbor
distances**. Calibrated on the **PSMA-specific BindingDB fine-tuning set** — that's
the domain the final predictions need to be trustworthy within, not the broad ChEMBL
pretraining set.


In [ ]:
def fps_to_array(fps) -> np.ndarray:
    arr = np.zeros((len(fps), fps[0].GetNumBits()), dtype=np.uint8)
    for i, fp in enumerate(fps):
        DataStructs.ConvertToNumpyArray(fp, arr[i])
    return arr


def bulk_min_tanimoto_dissimilarity(query_fps, ref_fps) -> np.ndarray:
    out = np.empty(len(query_fps))
    for i, qfp in enumerate(query_fps):
        sims = DataStructs.BulkTanimotoSimilarity(qfp, ref_fps)
        out[i] = 1.0 - max(sims)
    return out


def train_internal_tanimoto_dissimilarity(train_fps) -> np.ndarray:
    out = np.empty(len(train_fps))
    for i, qfp in enumerate(train_fps):
        sims = DataStructs.BulkTanimotoSimilarity(qfp, train_fps)
        sims[i] = -1
        out[i] = 1.0 - max(sims)
    return out


class ApplicabilityDomain:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.scaler: Optional[StandardScaler] = None
        self.pca: Optional[PCA] = None
        self.thresholds: dict = {}

    def fit(self, train_fps, train_desc: np.ndarray):
        self.scaler = StandardScaler().fit(np.nan_to_num(train_desc))
        train_desc_scaled = self.scaler.transform(np.nan_to_num(train_desc))

        n_comp = min(self.cfg.latent_pca_components, train_desc_scaled.shape[1], train_desc_scaled.shape[0] - 1)
        self.pca = PCA(n_components=max(n_comp, 1), random_state=self.cfg.random_state).fit(train_desc_scaled)
        train_latent = self.pca.transform(train_desc_scaled)

        tanimoto_internal = train_internal_tanimoto_dissimilarity(train_fps)
        self.thresholds["tanimoto"] = np.percentile(tanimoto_internal, self.cfg.ad_percentile_threshold)

        nn_desc_l1 = NearestNeighbors(n_neighbors=2, p=1).fit(train_desc_scaled)
        d_l1, _ = nn_desc_l1.kneighbors(train_desc_scaled)
        self.thresholds["descriptor_manhattan"] = np.percentile(d_l1[:, 1], self.cfg.ad_percentile_threshold)

        nn_latent_l1 = NearestNeighbors(n_neighbors=2, p=1).fit(train_latent)
        d_lat, _ = nn_latent_l1.kneighbors(train_latent)
        self.thresholds["latent_manhattan"] = np.percentile(d_lat[:, 1], self.cfg.ad_percentile_threshold)

        k = min(self.cfg.unfamiliarity_k + 1, len(train_desc_scaled))
        nn_unfam = NearestNeighbors(n_neighbors=k).fit(train_desc_scaled)
        d_unfam, _ = nn_unfam.kneighbors(train_desc_scaled)
        unfam_internal = d_unfam[:, 1:].mean(axis=1)
        self.thresholds["unfamiliarity"] = np.percentile(unfam_internal, self.cfg.ad_percentile_threshold)

        self._train_desc_scaled = train_desc_scaled
        self._train_latent = train_latent
        self._nn_desc_l1 = NearestNeighbors(n_neighbors=1, p=1).fit(train_desc_scaled)
        self._nn_latent_l1 = NearestNeighbors(n_neighbors=1, p=1).fit(train_latent)
        self._nn_unfam = NearestNeighbors(n_neighbors=min(self.cfg.unfamiliarity_k, len(train_desc_scaled))).fit(
            train_desc_scaled
        )
        return self

    def score(self, screen_fps, screen_desc: np.ndarray, train_fps) -> pd.DataFrame:
        screen_desc_scaled = self.scaler.transform(np.nan_to_num(screen_desc))
        screen_latent = self.pca.transform(screen_desc_scaled)

        tanimoto_diss = bulk_min_tanimoto_dissimilarity(screen_fps, train_fps)
        desc_manhattan, _ = self._nn_desc_l1.kneighbors(screen_desc_scaled)
        latent_manhattan, _ = self._nn_latent_l1.kneighbors(screen_latent)
        unfam_d, _ = self._nn_unfam.kneighbors(screen_desc_scaled)
        unfamiliarity = unfam_d.mean(axis=1)

        out = pd.DataFrame({
            "tanimoto_dissimilarity": tanimoto_diss,
            "descriptor_manhattan": desc_manhattan[:, 0],
            "latent_manhattan": latent_manhattan[:, 0],
            "unfamiliarity": unfamiliarity,
        })
        out["tanimoto_OOD"] = out["tanimoto_dissimilarity"] > self.thresholds["tanimoto"]
        out["descriptor_OOD"] = out["descriptor_manhattan"] > self.thresholds["descriptor_manhattan"]
        out["latent_OOD"] = out["latent_manhattan"] > self.thresholds["latent_manhattan"]
        out["unfamiliarity_OOD"] = out["unfamiliarity"] > self.thresholds["unfamiliarity"]
        out["n_flags_OOD"] = out[["tanimoto_OOD", "descriptor_OOD", "latent_OOD", "unfamiliarity_OOD"]].sum(axis=1)
        out["consensus_OOD"] = out["n_flags_OOD"] >= 3
        return out


ad = ApplicabilityDomain(CONFIG).fit(train_fps, train_desc)
ad_scores = ad.score(screen_fps, screen_desc, train_fps)
print("AD thresholds (95th percentile of the PSMA training set's own internal NN distances):")
for k, v in ad.thresholds.items():
    print(f"  {k}: {v:.4f}")
ad_scores.describe()


## 6b. Part B.2b — Which feature set actually works best here?

Both papers found that the best choice of input representation (fingerprints only,
descriptors only, or both together) is target-dependent rather than universal — the
KPLS paper found fingerprints alone beat descriptors alone on 9/10 targets, but the
*combination* won outright for one of them. Rather than assume `CONFIG.feature_set`
is right, a fast default-hyperparameter LightGBM is cross-validated on the PSMA
training data under all three options (scored on `pIC50`, the primary endpoint), and
whichever wins becomes the feature set used for every model built from here on.


In [ ]:
def build_design_matrix(fps, desc: np.ndarray, scaler: StandardScaler, feature_set: str) -> np.ndarray:
    desc_scaled = scaler.transform(np.nan_to_num(desc))
    if feature_set == "descriptors_only":
        return desc_scaled
    fp_arr = fps_to_array(fps)
    if feature_set == "fp_only":
        return fp_arr
    if feature_set == "fp_plus_descriptors":
        return np.hstack([fp_arr, desc_scaled])
    raise ValueError(f"Unknown feature_set '{feature_set}'.")


_feature_set_options = ["fp_only", "descriptors_only", "fp_plus_descriptors"]
_quick_scaler = StandardScaler().fit(np.nan_to_num(train_desc))
_kf_fs = KFold(n_splits=CONFIG.cv_folds, shuffle=True, random_state=CONFIG.random_state)

_fs_scores = {}
_y_probe = train_df[CONFIG.endpoint_names[0]].values.astype(np.float32)
_mask_probe = np.isfinite(_y_probe)
for fs in _feature_set_options:
    X_probe = build_design_matrix(train_fps, train_desc, _quick_scaler, fs)[_mask_probe]
    y_probe = _y_probe[_mask_probe]
    scores = []
    for tr_idx, te_idx in _kf_fs.split(X_probe):
        m = LGBMRegressor(random_state=CONFIG.random_state, verbose=-1, n_estimators=150, n_jobs=-1)
        m.fit(X_probe[tr_idx], y_probe[tr_idx])
        scores.append(r2_score(y_probe[te_idx], m.predict(X_probe[te_idx])))
    _fs_scores[fs] = float(np.mean(scores))
    print(f"  {fs:>22s}: mean CV R2 = {_fs_scores[fs]:.3f}")

_best_feature_set = max(_fs_scores, key=_fs_scores.get)
print(f"\nBest feature set for {CONFIG.endpoint_names[0]}: '{_best_feature_set}' "
      f"(CONFIG default was '{CONFIG.feature_set}')")
CONFIG.feature_set = _best_feature_set


## 7. Part B.3 — Design matrices

One shared feature scaler (`feature_scaler`, fit on the **broad ChEMBL pretraining
set**, since that's the larger and more chemically diverse sample) is reused to build
the model-input matrix for all three sets. Using the same scaler everywhere is what
lets the pretrained trunk's weights be meaningfully reused during fine-tuning — the
fine-tuning and screening inputs live in the exact same normalized feature space the
trunk was pretrained on. This is separate from `ad.scaler` above, which is
intentionally calibrated only on the PSMA set for domain-membership scoring.
`CONFIG.feature_set` here is whatever Part B.2b just selected.


In [ ]:
feature_scaler = StandardScaler().fit(np.nan_to_num(pretrain_desc))

X_pretrain = build_design_matrix(pretrain_fps, pretrain_desc, feature_scaler, CONFIG.feature_set)
y_pretrain = pretrain_df[CONFIG.chembl_pretrain_endpoint_name].values.astype(np.float32)

X_train_full = build_design_matrix(train_fps, train_desc, feature_scaler, CONFIG.feature_set)
Y_train_full = train_df[list(CONFIG.endpoint_names)].values.astype(np.float32)  # NaN where missing

X_screen = build_design_matrix(screen_fps, screen_desc, feature_scaler, CONFIG.feature_set)

X_tr, X_te, Y_tr, Y_te, idx_tr, idx_te = train_test_split(
    X_train_full, Y_train_full, np.arange(len(Y_train_full)),
    test_size=CONFIG.test_size, random_state=CONFIG.random_state
)

print(f"Feature set: '{CONFIG.feature_set}' -> {X_train_full.shape[1]} input dimensions")
print(f"ChEMBL pretraining: {X_pretrain.shape}")
print(f"PSMA fine-tuning — train: {X_tr.shape}, held-out test: {X_te.shape}")
for i, ep in enumerate(CONFIG.endpoint_names):
    print(f"  {ep}: {np.isfinite(Y_tr[:, i]).sum()} labeled in train, {np.isfinite(Y_te[:, i]).sum()} in test")


## 8. Part C.1 — Multi-task transfer-learning neural network

A small MLP **trunk** shared across tasks, with:

- a **pretrain head** (single output) used only while pretraining on the broad
  ChEMBL set (generic potency regression — a standard proxy target for learning a
  general structure→bioactivity representation), and
- two **task heads** (pIC50, pKi) used during fine-tuning on the PSMA set.

Fine-tuning **loads the pretrained trunk's weights** (the actual "transfer" step),
then trains with a **masked MSE loss** so a compound missing one of the two labels
still contributes through the other, and uses **gradual unfreezing**: the trunk is
frozen for the first `nn_freeze_epochs` (only the new heads adapt to PSMA data),
then unfrozen so the whole network can fine-tune together — a standard transfer-
learning trick that avoids catastrophically overwriting the pretrained
representation before the new heads have caught up.

**MC-dropout** (keeping dropout active at prediction time and averaging many
stochastic forward passes) gives this model a built-in uncertainty estimate, used
later alongside the GBM baseline's bootstrap uncertainty.

Every function below takes a `device` argument (defaulting to the `DEVICE` detected
in Setup), so pretraining, fine-tuning, and MC-dropout inference all run on GPU
automatically when one's available.


In [ ]:
class MultiTaskMLP(nn.Module):
    def __init__(self, n_in: int, hidden_dims=(256, 128), dropout: float = 0.2, n_tasks: int = 2):
        super().__init__()
        layers = []
        d = n_in
        for h in hidden_dims:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        self.trunk = nn.Sequential(*layers)
        self.trunk_out_dim = d
        self.pretrain_head = nn.Linear(d, 1)
        self.task_heads = nn.ModuleList([nn.Linear(d, 1) for _ in range(n_tasks)])

    def forward_pretrain(self, x):
        return self.pretrain_head(self.trunk(x)).squeeze(-1)

    def forward_tasks(self, x):
        z = self.trunk(x)
        return torch.cat([h(z) for h in self.task_heads], dim=1)


def masked_mse(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    '''target may contain NaN; loss is averaged only over the non-NaN entries, per batch,
    across all tasks at once — this is what lets partially-labeled rows contribute.'''
    mask = ~torch.isnan(target)
    if mask.sum() == 0:
        return torch.tensor(0.0, requires_grad=True, device=pred.device)
    diff = (pred - torch.nan_to_num(target, nan=0.0)) ** 2
    return (diff * mask).sum() / mask.sum().clamp(min=1)


def pretrain_trunk(X, y, hidden_dims, dropout, lr, weight_decay, epochs, batch_size, seed=0,
                    device=DEVICE) -> MultiTaskMLP:
    torch.manual_seed(seed)
    model = MultiTaskMLP(X.shape[1], hidden_dims=hidden_dims, dropout=dropout,
                          n_tasks=len(CONFIG.endpoint_names)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    Xt = torch.tensor(X, dtype=torch.float32, device=device)
    yt = torch.tensor(y, dtype=torch.float32, device=device)
    n = len(X)
    for _ in tqdm(range(epochs), desc="Pretraining trunk on broad ChEMBL data"):
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            loss = nn.functional.mse_loss(model.forward_pretrain(Xt[idx]), yt[idx])
            loss.backward()
            opt.step()
    return model


def finetune_multitask(pretrained_trunk_state, X, Y, hidden_dims, dropout, lr, weight_decay,
                        freeze_epochs, total_epochs, batch_size, seed=0, device=DEVICE) -> MultiTaskMLP:
    '''Loads a pretrained trunk (if given — this is the transfer-learning step), freezes it
    for `freeze_epochs` while the new task heads adapt, then unfreezes for the remainder
    (gradual unfreezing) to fine-tune the whole network jointly on the multi-task,
    partially-masked PSMA labels. Runs on `device` (GPU if available) end to end.'''
    torch.manual_seed(seed)
    model = MultiTaskMLP(X.shape[1], hidden_dims=hidden_dims, dropout=dropout, n_tasks=Y.shape[1])
    if pretrained_trunk_state is not None:
        model.trunk.load_state_dict(pretrained_trunk_state)
    model = model.to(device)
    Xt = torch.tensor(X, dtype=torch.float32, device=device)
    Yt = torch.tensor(Y, dtype=torch.float32, device=device)
    n = len(X)

    for ep in range(total_epochs):
        frozen = ep < freeze_epochs
        for p in model.trunk.parameters():
            p.requires_grad = not frozen
        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            loss = masked_mse(model.forward_tasks(Xt[idx]), Yt[idx])
            loss.backward()
            opt.step()
    return model


def predict_with_mc_dropout(model: MultiTaskMLP, X, n_passes: int, device=DEVICE) -> Tuple[np.ndarray, np.ndarray]:
    '''Returns (mean, std) per task, each shape (n_samples, n_tasks), from `n_passes`
    stochastic forward passes with dropout kept active. Moves `model` and the input onto
    `device` (GPU if available); results are brought back to plain numpy on CPU.'''
    model = model.to(device)
    Xt = torch.tensor(X, dtype=torch.float32, device=device)
    model.train()
    with torch.no_grad():
        preds = torch.stack([model.forward_tasks(Xt) for _ in range(n_passes)], dim=0).cpu().numpy()
    model.eval()
    return preds.mean(axis=0), preds.std(axis=0)


print(f"Multi-task transfer-learning model defined: MultiTaskMLP, pretrain_trunk, "
      f"finetune_multitask, predict_with_mc_dropout. Training device: {DEVICE}.")


## 9. Part C.2 — Baselines: LightGBM and Kernel-PLS (trained directly on PSMA data, per task)

Two non-transfer-learning comparators, trained directly on the (masked-to-that-task's-
labeled-rows) BindingDB PSMA data:

- **LightGBM** — one `LGBMRegressor` per endpoint.
- **Kernel-PLS (KPLS)** — following An, Sherman & Dixon (*J. Chem. Inf. Model.* 2013):
  a Gaussian-kernel approximation (`Nystroem`) feeding a `PLSRegression`. Their paper
  found KPLS on fingerprints alone frequently *beats* larger descriptor-based models
  for congeneric binding-affinity series — exactly our situation with PSMA — so it's
  included as a genuinely competitive third model family, not a token baseline.

Whichever of the three (NN transfer, LightGBM, KPLS) wins on held-out data is used —
optionally blended with the others — for the final screening predictions.


In [ ]:
def fit_lgbm_for_task(X: np.ndarray, y_masked: np.ndarray, params: dict, seed: int = 42) -> LGBMRegressor:
    mask = np.isfinite(y_masked)
    model = LGBMRegressor(random_state=seed, verbose=-1, n_jobs=-1, **params)
    model.fit(X[mask], y_masked[mask])
    return model


def bootstrap_lgbm_ensemble(X: np.ndarray, y_masked: np.ndarray, params: dict, n_bootstrap: int,
                             seed: int = 42) -> List[LGBMRegressor]:
    '''Bootstrap-resampled ensemble of the same tuned LightGBM config, used for
    prediction-uncertainty estimation (works for any regressor, unlike relying on a
    single Random-Forest's internal tree variance).'''
    mask = np.isfinite(y_masked)
    Xs, ys = X[mask], y_masked[mask]
    rng = np.random.RandomState(seed)
    models = []
    for b in range(n_bootstrap):
        idx = rng.randint(0, len(Xs), size=len(Xs))
        m = LGBMRegressor(random_state=seed + b, verbose=-1, n_jobs=-1, **params)
        m.fit(Xs[idx], ys[idx])
        models.append(m)
    return models


def make_kpls_pipeline(n_components: int, gamma: float, seed: int = 42) -> Pipeline:
    '''Gaussian-kernel PLS via a Nystroem feature-map approximation (this is the "direct
    kernel PLS" trick from An/Sherman/Dixon 2013: approximate the kernel matrix with a
    low-rank map, then run ordinary PLS in that mapped space).'''
    n_comp = max(1, min(n_components, 50))
    return Pipeline([
        ("nystroem", Nystroem(kernel="rbf", gamma=gamma, n_components=200, random_state=seed)),
        ("pls", PLSRegression(n_components=n_comp)),
    ])


def fit_kpls_for_task(X: np.ndarray, y_masked: np.ndarray, params: dict, seed: int = 42) -> Pipeline:
    mask = np.isfinite(y_masked)
    model = make_kpls_pipeline(params["n_components"], params["gamma"], seed=seed)
    model.fit(X[mask], y_masked[mask])
    return model


def kpls_predict(model: Pipeline, X: np.ndarray) -> np.ndarray:
    '''PLSRegression.predict returns shape (n, 1); flatten to match the other models.'''
    return np.ravel(model.predict(X))


def bootstrap_kpls_ensemble(X: np.ndarray, y_masked: np.ndarray, params: dict, n_bootstrap: int,
                             seed: int = 42) -> List[Pipeline]:
    mask = np.isfinite(y_masked)
    Xs, ys = X[mask], y_masked[mask]
    rng = np.random.RandomState(seed)
    models = []
    for b in range(n_bootstrap):
        idx = rng.randint(0, len(Xs), size=len(Xs))
        m = make_kpls_pipeline(params["n_components"], params["gamma"], seed=seed + b)
        m.fit(Xs[idx], ys[idx])
        models.append(m)
    return models


print("Baseline helpers defined: fit_lgbm_for_task, bootstrap_lgbm_ensemble, "
      "fit_kpls_for_task, kpls_predict, bootstrap_kpls_ensemble.")


## 10. Part D.1 — Hyperparameter tuning: LightGBM & KPLS

Each task's LightGBM is tuned two ways on the same K-fold CV splits (KPLS, being
cheap to fit, is tuned with `GridSearchCV` alone):

- **`GridSearchCV`** — exhaustive search over a hand-picked grid.
- **`BayesSearchCV`** (scikit-optimize) — Bayesian optimization over continuous/
  integer search spaces, usually finding as-good-or-better configurations in far
  fewer fits than the grid.

Candidates are ranked by the **AutoQSAR score** rather than raw CV test R²:

```
score = accuracy_test * (1 - |accuracy_train - accuracy_test|)
```

(Dixon et al., *Future Med. Chem.* 2016). This rewards configurations that generalize
— high test accuracy *and* a small train/test gap — and penalizes the kind of
overfit config that racks up a great CV test score by memorizing a small training
fold. `GridSearchCV`/`BayesSearchCV`'s own `return_train_score=True` gives us the
train-fold accuracy needed to compute it, for free, alongside the search itself.

Rows without a label for a given task are dropped before tuning *that* task's model
(GBM/KPLS regressors can't consume `NaN` targets the way the masked-loss NN can).


In [ ]:
def autoqsar_score(accuracy_train: float, accuracy_test: float) -> float:
    '''AutoQSAR's model-quality score (Dixon et al. 2016, Eq. 1): rewards models that are
    accurate on *both* train and test, and penalizes a large train/test gap. If the test
    accuracy itself is negative (predictions worse than predicting the mean), the score
    is just that negative value, per the paper's own handling of that edge case.'''
    if accuracy_test < 0:
        return accuracy_test
    return accuracy_test * (1.0 - abs(accuracy_train - accuracy_test))


def pick_best_by_autoqsar_score(cv_results: dict) -> Tuple[int, float, float, float]:
    '''Given an sklearn-style `cv_results_` dict (from GridSearchCV/BayesSearchCV fit with
    return_train_score=True), returns (best_index, best_autoqsar_score, train_r2, test_r2).'''
    train_scores = cv_results["mean_train_score"]
    test_scores = cv_results["mean_test_score"]
    aq_scores = [autoqsar_score(tr, te) for tr, te in zip(train_scores, test_scores)]
    best_idx = int(np.argmax(aq_scores))
    return best_idx, aq_scores[best_idx], float(train_scores[best_idx]), float(test_scores[best_idx])


GRID_PARAMS = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.01, 0.05, 0.1],
    "num_leaves": [15, 31, 63],
    "max_depth": [-1, 6, 10],
}
BAYES_SPACE = {
    "n_estimators": Integer(50, 500),
    "learning_rate": Real(0.005, 0.3, prior="log-uniform"),
    "num_leaves": Integer(7, 63),
    "max_depth": Integer(3, 12),
    "min_child_samples": Integer(5, 40),
}
KPLS_GRID_PARAMS = {
    "n_components": [2, 4, 6, 8, 12],
    "gamma": [1.0 / X_tr.shape[1], 0.01, 0.05, 0.1],
}

gbm_tuning_results, gbm_best_params = {}, {}
kpls_tuning_results, kpls_best_params = {}, {}

for t, ep in enumerate(CONFIG.endpoint_names):
    y_task = Y_tr[:, t]
    mask = np.isfinite(y_task)
    Xs, ys = X_tr[mask], y_task[mask]
    kf = KFold(n_splits=CONFIG.cv_folds, shuffle=True, random_state=CONFIG.random_state)
    print(f"\n=== Tuning models for {ep} ({mask.sum()} labeled training rows) ===")

    # --- LightGBM: GridSearchCV vs BayesSearchCV, both scored with the AutoQSAR metric ---
    candidates = {}
    if CONFIG.run_gridsearch:
        gcv = GridSearchCV(LGBMRegressor(random_state=CONFIG.random_state, verbose=-1, n_jobs=-1),
                            GRID_PARAMS, cv=kf, scoring="r2", n_jobs=-1, return_train_score=True)
        gcv.fit(Xs, ys)
        idx, aq, tr_r2, te_r2 = pick_best_by_autoqsar_score(gcv.cv_results_)
        candidates["GridSearchCV"] = (aq, gcv.cv_results_["params"][idx], tr_r2, te_r2)
        print(f"  LightGBM GridSearchCV  best AutoQSAR score={aq:.3f} (train R2={tr_r2:.3f}, "
              f"test R2={te_r2:.3f})  params={gcv.cv_results_['params'][idx]}")

    if CONFIG.run_bayes_search:
        bcv = BayesSearchCV(LGBMRegressor(random_state=CONFIG.random_state, verbose=-1, n_jobs=-1),
                             BAYES_SPACE, n_iter=CONFIG.bayes_search_n_iter, cv=kf, scoring="r2",
                             n_jobs=-1, random_state=CONFIG.random_state, return_train_score=True)
        bcv.fit(Xs, ys)
        idx, aq, tr_r2, te_r2 = pick_best_by_autoqsar_score(bcv.cv_results_)
        candidates["BayesSearchCV"] = (aq, dict(bcv.cv_results_["params"][idx]), tr_r2, te_r2)
        print(f"  LightGBM BayesSearchCV best AutoQSAR score={aq:.3f} (train R2={tr_r2:.3f}, "
              f"test R2={te_r2:.3f})  params={dict(bcv.cv_results_['params'][idx])}")

    winner = max(candidates, key=lambda k: candidates[k][0])
    gbm_tuning_results[ep] = candidates
    gbm_best_params[ep] = candidates[winner][1]
    print(f"  -> LightGBM winner for {ep}: {winner} (AutoQSAR score={candidates[winner][0]:.3f})")

    # --- KPLS: GridSearchCV only (cheap model, small search space) ---
    kcv = GridSearchCV(make_kpls_pipeline(4, 0.01), {"nystroem__gamma": KPLS_GRID_PARAMS["gamma"],
                                                      "pls__n_components": KPLS_GRID_PARAMS["n_components"]},
                        cv=kf, scoring="r2", n_jobs=-1, return_train_score=True)
    kcv.fit(Xs, ys)
    idx, aq, tr_r2, te_r2 = pick_best_by_autoqsar_score(kcv.cv_results_)
    best_kpls_raw = kcv.cv_results_["params"][idx]
    kpls_best_params[ep] = {"n_components": best_kpls_raw["pls__n_components"], "gamma": best_kpls_raw["nystroem__gamma"]}
    kpls_tuning_results[ep] = {"GridSearchCV": (aq, kpls_best_params[ep], tr_r2, te_r2)}
    print(f"  KPLS GridSearchCV     best AutoQSAR score={aq:.3f} (train R2={tr_r2:.3f}, "
          f"test R2={te_r2:.3f})  params={kpls_best_params[ep]}")

print("\nFinal tuned params:")
print("  LightGBM:", gbm_best_params)
print("  KPLS:", kpls_best_params)


## 11. Part D.2 — Hyperparameter tuning: NN fine-tuning (Optuna, TPE sampler)

`GridSearchCV`/`BayesSearchCV` don't apply cleanly to the pretrain→fine-tune network
(it isn't a plain sklearn estimator), so its architecture and fine-tuning
hyperparameters are tuned with **Optuna**'s Tree-structured Parzen Estimator — a
Bayesian search over hidden-layer sizes, dropout, learning rate, weight decay,
freeze-epoch schedule, and batch size. Each trial re-pretrains (cheaply, on a
reduced epoch budget) and fine-tunes on a train/validation split of the PSMA set,
scored by mean held-out R² across both tasks (masked to whichever labels each
validation row actually has).


In [ ]:
X_opt_tr, X_opt_val, Y_opt_tr, Y_opt_val = train_test_split(
    X_tr, Y_tr, test_size=1.0 / CONFIG.optuna_cv_folds, random_state=CONFIG.random_state
)

# A cheap pretrain pass reused as the base for every trial that uses default-sized hidden dims;
# trials that request a different architecture pretrain their own (still on a reduced budget).
_default_hidden = tuple(CONFIG.nn_hidden_dims)
_quick_pretrain_epochs = max(5, CONFIG.nn_pretrain_epochs // 4)
_base_pretrained = pretrain_trunk(
    X_pretrain, y_pretrain, hidden_dims=_default_hidden, dropout=CONFIG.nn_dropout,
    lr=CONFIG.nn_lr, weight_decay=CONFIG.nn_weight_decay, epochs=_quick_pretrain_epochs,
    batch_size=CONFIG.nn_pretrain_batch_size, seed=CONFIG.random_state,
)


def optuna_objective(trial: "optuna.Trial") -> float:
    h1 = trial.suggest_categorical("h1", [64, 128, 256])
    h2 = trial.suggest_categorical("h2", [32, 64, 128])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    freeze_epochs = trial.suggest_int("freeze_epochs", 0, 30)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    hidden_dims = (h1, h2)
    trunk_state = (_base_pretrained.trunk.state_dict() if hidden_dims == _default_hidden
                   else pretrain_trunk(X_pretrain, y_pretrain, hidden_dims=hidden_dims, dropout=dropout,
                                       lr=CONFIG.nn_lr, weight_decay=weight_decay,
                                       epochs=_quick_pretrain_epochs, batch_size=CONFIG.nn_pretrain_batch_size,
                                       seed=CONFIG.random_state).trunk.state_dict())

    model = finetune_multitask(trunk_state, X_opt_tr, Y_opt_tr, hidden_dims=hidden_dims, dropout=dropout,
                                lr=lr, weight_decay=weight_decay, freeze_epochs=min(freeze_epochs, 30),
                                total_epochs=max(30, min(freeze_epochs + 20, CONFIG.nn_finetune_total_epochs)),
                                batch_size=batch_size, seed=CONFIG.random_state)
    with torch.no_grad():
        pred_val = model.forward_tasks(torch.tensor(X_opt_val, dtype=torch.float32, device=DEVICE)).cpu().numpy()

    scores = []
    for t in range(Y_opt_val.shape[1]):
        mask = np.isfinite(Y_opt_val[:, t])
        if mask.sum() > 3:
            scores.append(r2_score(Y_opt_val[mask, t], pred_val[mask, t]))
    return float(np.mean(scores)) if scores else -1.0


if CONFIG.run_optuna:
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=CONFIG.random_state))
    study.optimize(optuna_objective, n_trials=CONFIG.optuna_n_trials, show_progress_bar=False)
    nn_best_params = study.best_params
    print(f"Optuna best mean validation R2 = {study.best_value:.3f}")
    print(f"Optuna best params: {nn_best_params}")
else:
    nn_best_params = {"h1": CONFIG.nn_hidden_dims[0], "h2": CONFIG.nn_hidden_dims[1], "dropout": CONFIG.nn_dropout,
                       "lr": CONFIG.nn_finetune_lr, "weight_decay": CONFIG.nn_weight_decay,
                       "freeze_epochs": CONFIG.nn_freeze_epochs, "batch_size": CONFIG.nn_batch_size}
    print("Optuna disabled — using CONFIG defaults:", nn_best_params)

nn_best_params


## 12. Part E.1 — Train the final models

Using the winning hyperparameters from Part D:

- **NN**: pretrain the trunk on the *full* ChEMBL set with the tuned architecture,
  then fine-tune on the PSMA **training split** (`X_tr`/`Y_tr`) for held-out
  evaluation, and separately on the **full** PSMA set for the model that actually
  ships/predicts on the screening library.
- **GBM and KPLS**: one tuned model per task per family, same
  train-split-then-full-refit pattern, plus a bootstrap ensemble (on the full data)
  for uncertainty.


In [ ]:
nn_hidden_dims_final = (nn_best_params["h1"], nn_best_params["h2"])

# --- pretrain trunk on the FULL ChEMBL set with tuned architecture ---
pretrained_final = pretrain_trunk(
    X_pretrain, y_pretrain, hidden_dims=nn_hidden_dims_final, dropout=nn_best_params["dropout"],
    lr=CONFIG.nn_lr, weight_decay=nn_best_params["weight_decay"], epochs=CONFIG.nn_pretrain_epochs,
    batch_size=CONFIG.nn_pretrain_batch_size, seed=CONFIG.random_state,
)

# --- fine-tune on the PSMA TRAIN split, for honest held-out evaluation ---
nn_model_eval = finetune_multitask(
    pretrained_final.trunk.state_dict(), X_tr, Y_tr, hidden_dims=nn_hidden_dims_final,
    dropout=nn_best_params["dropout"], lr=nn_best_params["lr"], weight_decay=nn_best_params["weight_decay"],
    freeze_epochs=nn_best_params["freeze_epochs"], total_epochs=CONFIG.nn_finetune_total_epochs,
    batch_size=nn_best_params["batch_size"], seed=CONFIG.random_state,
)

# --- fine-tune on the FULL PSMA set, for the deployed / screening-time model ---
nn_model_deploy = finetune_multitask(
    pretrained_final.trunk.state_dict(), X_train_full, Y_train_full, hidden_dims=nn_hidden_dims_final,
    dropout=nn_best_params["dropout"], lr=nn_best_params["lr"], weight_decay=nn_best_params["weight_decay"],
    freeze_epochs=nn_best_params["freeze_epochs"], total_epochs=CONFIG.nn_finetune_total_epochs,
    batch_size=nn_best_params["batch_size"], seed=CONFIG.random_state,
)

# --- tuned LightGBM & KPLS per task: fit on TRAIN split (eval) and on FULL data (deploy) ---
gbm_models_eval, gbm_models_deploy, gbm_bootstrap_ensembles = {}, {}, {}
kpls_models_eval, kpls_models_deploy, kpls_bootstrap_ensembles = {}, {}, {}
for t, ep in enumerate(CONFIG.endpoint_names):
    gp = gbm_best_params[ep]
    gbm_models_eval[ep] = fit_lgbm_for_task(X_tr, Y_tr[:, t], gp, seed=CONFIG.random_state)
    gbm_models_deploy[ep] = fit_lgbm_for_task(X_train_full, Y_train_full[:, t], gp, seed=CONFIG.random_state)
    gbm_bootstrap_ensembles[ep] = bootstrap_lgbm_ensemble(
        X_train_full, Y_train_full[:, t], gp, n_bootstrap=CONFIG.n_bootstrap, seed=CONFIG.random_state
    )

    kp = kpls_best_params[ep]
    kpls_models_eval[ep] = fit_kpls_for_task(X_tr, Y_tr[:, t], kp, seed=CONFIG.random_state)
    kpls_models_deploy[ep] = fit_kpls_for_task(X_train_full, Y_train_full[:, t], kp, seed=CONFIG.random_state)
    kpls_bootstrap_ensembles[ep] = bootstrap_kpls_ensemble(
        X_train_full, Y_train_full[:, t], kp, n_bootstrap=CONFIG.n_bootstrap, seed=CONFIG.random_state
    )

print("Final NN, LightGBM, and KPLS models trained (eval + deploy + bootstrap where applicable).")


## 13. Part E.2 — Held-out evaluation, blend, and robustness

Three things happen here:

1. **Score all three model families** (NN transfer, LightGBM, KPLS) on the held-out
   PSMA test split, then find the blend weights `w_NN·NN + w_GBM·GBM + w_KPLS·KPLS`
   (weights on the simplex, i.e. non-negative and summing to 1) that minimize
   held-out RMSE per task — a 3-way generalization of simple pairwise blending.
2. **Repeat the split `CONFIG.n_eval_repeats` times** (AutoQSAR uses 50 random
   75/25 splits rather than trusting one partition, precisely because a single small
   test set's composition can swing the reported R²/Q² — their own BCF and
   solubility case studies are examples of exactly that). Reusing the already-
   pretrained trunk keeps each repeat cheap: only fine-tuning (not pretraining) is
   redone per split.
3. **Collect (predicted-uncertainty, actual-error) pairs across every repeat**, for
   the calibration check in Part E.2b — this needs uncertainty estimates on
   *held-out* compounds, not just the screening library, which is why it's computed
   here rather than only at screening time.


In [ ]:
def optimize_blend_weights(preds: Dict[str, np.ndarray], y_true: np.ndarray) -> Dict[str, float]:
    '''Finds non-negative weights (summing to 1) over an arbitrary number of model
    predictions that minimize RMSE against y_true — a simplex-constrained least-squares
    blend, generalizing simple two-model weight search to any number of models.'''
    from scipy.optimize import minimize

    names = list(preds.keys())
    P = np.column_stack([preds[n] for n in names])
    n_models = P.shape[1]

    def objective(w):
        return rmse(y_true, P @ w)

    w0 = np.full(n_models, 1.0 / n_models)
    cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, 1.0)] * n_models
    res = minimize(objective, w0, method="SLSQP", bounds=bounds, constraints=cons)
    w = res.x if res.success else w0
    w = np.clip(w, 0, None)
    w = w / w.sum()
    return {n: float(wi) for n, wi in zip(names, w)}


with torch.no_grad():
    nn_test_pred = nn_model_eval.to(DEVICE).forward_tasks(
        torch.tensor(X_te, dtype=torch.float32, device=DEVICE)
    ).cpu().numpy()

eval_rows = []
blend_weights = {}
for t, ep in enumerate(CONFIG.endpoint_names):
    mask = np.isfinite(Y_te[:, t])
    y_true = Y_te[mask, t]
    preds = {
        "nn": nn_test_pred[mask, t],
        "gbm": gbm_models_eval[ep].predict(X_te[mask]),
        "kpls": kpls_predict(kpls_models_eval[ep], X_te[mask]),
    }
    w = optimize_blend_weights(preds, y_true)
    blend_weights[ep] = w
    blend_pred = sum(w[n] * preds[n] for n in preds)

    for label, pred in [("NN (transfer)", preds["nn"]), ("LightGBM", preds["gbm"]), ("KPLS", preds["kpls"]),
                         (f"Blend (w={ {k: round(v, 2) for k, v in w.items()} })", blend_pred)]:
        eval_rows.append({
            "endpoint": ep, "model": label, "n_test": int(mask.sum()),
            "R2": r2_score(y_true, pred), "RMSE": rmse(y_true, pred), "MAE": mean_absolute_error(y_true, pred),
        })

eval_df = pd.DataFrame(eval_rows)
eval_df


In [ ]:
fig, axes = plt.subplots(1, len(CONFIG.endpoint_names), figsize=(6 * len(CONFIG.endpoint_names), 5))
if len(CONFIG.endpoint_names) == 1:
    axes = [axes]
for ax, (t, ep) in zip(axes, enumerate(CONFIG.endpoint_names)):
    mask = np.isfinite(Y_te[:, t])
    y_true = Y_te[mask, t]
    preds = {"nn": nn_test_pred[mask, t], "gbm": gbm_models_eval[ep].predict(X_te[mask]),
             "kpls": kpls_predict(kpls_models_eval[ep], X_te[mask])}
    w = blend_weights[ep]
    blend_pred = sum(w[n] * preds[n] for n in preds)
    ax.scatter(y_true, preds["nn"], alpha=0.5, label="NN (transfer)", s=25)
    ax.scatter(y_true, preds["gbm"], alpha=0.5, label="LightGBM", s=25)
    ax.scatter(y_true, preds["kpls"], alpha=0.5, label="KPLS", s=25)
    ax.scatter(y_true, blend_pred, alpha=0.8, label="Blend", s=30, marker="x")
    all_vals = np.concatenate([y_true, preds["nn"], preds["gbm"], preds["kpls"]])
    lims = [all_vals.min(), all_vals.max()]
    ax.plot(lims, lims, "k--", alpha=0.5, linewidth=1)
    ax.set_xlabel(f"Experimental {ep}")
    ax.set_ylabel(f"Predicted {ep}")
    ax.set_title(f"{ep} — held-out test (n={mask.sum()})")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()


## 13b. Part E.2b — Repeated-split robustness and uncertainty calibration

**Robustness**: the single split above is re-drawn `CONFIG.n_eval_repeats` times
(fresh fine-tune of the already-pretrained NN trunk, fresh GBM/KPLS fits, same tuned
hyperparameters each time) and R²/RMSE/MAE are reported as mean ± std rather than a
single number — following the AutoQSAR paper's own use of 50 random splits for this
reason.

**Calibration** (following the KPLS paper's Figure 7 exactly): every held-out
compound across every repeat gets a predicted uncertainty (MC-dropout for the NN,
bootstrap ensembles for GBM/KPLS, blended in quadrature using the weights from Part
E.2) *and* a known absolute error. Binning by predicted uncertainty and plotting the
cumulative distribution of absolute error within each bin answers the question that
matters for the screening predictions later: **can the uncertainty estimate actually
be trusted to flag less-reliable predictions?**


In [ ]:
_N_BOOTSTRAP_REPEAT = min(10, CONFIG.n_bootstrap)
_N_MC_REPEAT = min(15, CONFIG.nn_mc_dropout_passes)

repeated_eval_rows = []
calibration_records = []   # (endpoint, model, predicted_std, abs_error)

for r in range(CONFIG.n_eval_repeats):
    seed_r = CONFIG.random_state + 1000 + r
    X_tr_r, X_te_r, Y_tr_r, Y_te_r = train_test_split(
        X_train_full, Y_train_full, test_size=CONFIG.test_size, random_state=seed_r
    )

    nn_r = finetune_multitask(
        pretrained_final.trunk.state_dict(), X_tr_r, Y_tr_r, hidden_dims=nn_hidden_dims_final,
        dropout=nn_best_params["dropout"], lr=nn_best_params["lr"], weight_decay=nn_best_params["weight_decay"],
        freeze_epochs=nn_best_params["freeze_epochs"], total_epochs=CONFIG.nn_finetune_total_epochs,
        batch_size=nn_best_params["batch_size"], seed=seed_r,
    )
    nn_mean_r, nn_std_r = predict_with_mc_dropout(nn_r, X_te_r, _N_MC_REPEAT)

    for t, ep in enumerate(CONFIG.endpoint_names):
        mask = np.isfinite(Y_te_r[:, t])
        if mask.sum() < 3:
            continue
        y_true = Y_te_r[mask, t]

        gbm_r = fit_lgbm_for_task(X_tr_r, Y_tr_r[:, t], gbm_best_params[ep], seed=seed_r)
        gbm_boot_r = bootstrap_lgbm_ensemble(X_tr_r, Y_tr_r[:, t], gbm_best_params[ep],
                                              n_bootstrap=_N_BOOTSTRAP_REPEAT, seed=seed_r)
        kpls_r = fit_kpls_for_task(X_tr_r, Y_tr_r[:, t], kpls_best_params[ep], seed=seed_r)
        kpls_boot_r = bootstrap_kpls_ensemble(X_tr_r, Y_tr_r[:, t], kpls_best_params[ep],
                                               n_bootstrap=_N_BOOTSTRAP_REPEAT, seed=seed_r)

        nn_pred = nn_mean_r[mask, t]
        gbm_pred = gbm_r.predict(X_te_r[mask])
        kpls_pred = kpls_predict(kpls_r, X_te_r[mask])
        w = blend_weights[ep]
        blend_pred = w["nn"] * nn_pred + w["gbm"] * gbm_pred + w["kpls"] * kpls_pred

        for label, pred in [("NN", nn_pred), ("GBM", gbm_pred), ("KPLS", kpls_pred), ("Blend", blend_pred)]:
            repeated_eval_rows.append({
                "repeat": r, "endpoint": ep, "model": label,
                "R2": r2_score(y_true, pred), "RMSE": rmse(y_true, pred), "MAE": mean_absolute_error(y_true, pred),
            })

        gbm_boot_std = np.stack([m.predict(X_te_r[mask]) for m in gbm_boot_r], axis=0).std(axis=0)
        kpls_boot_std = np.stack([kpls_predict(m, X_te_r[mask]) for m in kpls_boot_r], axis=0).std(axis=0)
        blend_std = np.sqrt((w["nn"] * nn_std_r[mask, t]) ** 2 + (w["gbm"] * gbm_boot_std) ** 2
                             + (w["kpls"] * kpls_boot_std) ** 2)
        abs_err = np.abs(blend_pred - y_true)
        for s, e in zip(blend_std, abs_err):
            calibration_records.append({"endpoint": ep, "predicted_std": s, "abs_error": e})

repeated_eval_df = pd.DataFrame(repeated_eval_rows)
calibration_df = pd.DataFrame(calibration_records)

repeated_eval_summary = (
    repeated_eval_df.groupby(["endpoint", "model"])[["R2", "RMSE", "MAE"]]
    .agg(["mean", "std"])
)
print(f"Repeated-split evaluation over {CONFIG.n_eval_repeats} random 80/20 splits:")
repeated_eval_summary


In [ ]:
fig, axes = plt.subplots(1, len(CONFIG.endpoint_names), figsize=(6 * len(CONFIG.endpoint_names), 5))
if len(CONFIG.endpoint_names) == 1:
    axes = [axes]
for ax, ep in zip(axes, CONFIG.endpoint_names):
    sub = calibration_df[calibration_df["endpoint"] == ep].copy()
    if len(sub) < CONFIG.n_calibration_bins * 3:
        ax.set_title(f"{ep}: not enough held-out points for calibration bins")
        continue
    sub["bin"] = pd.qcut(sub["predicted_std"], q=CONFIG.n_calibration_bins, duplicates="drop")
    for bin_label, group in sub.groupby("bin", observed=True):
        errs = np.sort(group["abs_error"].values)
        cum_prob = np.arange(1, len(errs) + 1) / len(errs)
        ax.plot(errs, cum_prob, label=f"std in {bin_label}", linewidth=1.5)
    ax.set_xlabel("Absolute error")
    ax.set_ylabel("Cumulative probability")
    ax.set_title(f"{ep}: error calibration by predicted uncertainty\n(low-uncertainty bins should sit up-and-left)")
    ax.legend(fontsize=6)
plt.tight_layout()
plt.show()
print("If the lowest-uncertainty bin's curve consistently sits above and to the left of the "
      "highest-uncertainty bin's curve, the blended uncertainty estimate is well-calibrated: "
      "predictions the model is confident about really are more accurate, exactly as found in "
      "the KPLS paper's own bootstrapping analysis (their Figure 7).")


## 14. Part E.3 — Screen the library

For each endpoint: blend the deployed NN's MC-dropout mean with the deployed GBM's
and KPLS's bootstrap-ensemble means using the held-out-tuned 3-way blend weights, and
combine their uncertainties in quadrature. Every prediction ships with its
Applicability-Domain flags from Part B.2 so out-of-domain predictions are clearly
marked as lower-confidence regardless of how tight the model's own (now
calibration-checked, per Part E.2b) uncertainty estimate looks.


In [ ]:
nn_screen_mean, nn_screen_std = predict_with_mc_dropout(nn_model_deploy, X_screen, CONFIG.nn_mc_dropout_passes)

screen_results = screen_df.copy().reset_index(drop=True)
for t, ep in enumerate(CONFIG.endpoint_names):
    gbm_preds = np.stack([m.predict(X_screen) for m in gbm_bootstrap_ensembles[ep]], axis=0)
    gbm_mean, gbm_std = gbm_preds.mean(axis=0), gbm_preds.std(axis=0)

    kpls_preds = np.stack([kpls_predict(m, X_screen) for m in kpls_bootstrap_ensembles[ep]], axis=0)
    kpls_mean, kpls_std = kpls_preds.mean(axis=0), kpls_preds.std(axis=0)

    w = blend_weights[ep]
    blend_mean = w["nn"] * nn_screen_mean[:, t] + w["gbm"] * gbm_mean + w["kpls"] * kpls_mean
    blend_std = np.sqrt((w["nn"] * nn_screen_std[:, t]) ** 2 + (w["gbm"] * gbm_std) ** 2
                         + (w["kpls"] * kpls_std) ** 2)

    screen_results[f"pred_{ep}_nn"] = nn_screen_mean[:, t]
    screen_results[f"pred_{ep}_gbm"] = gbm_mean
    screen_results[f"pred_{ep}_kpls"] = kpls_mean
    screen_results[f"pred_{ep}"] = blend_mean
    screen_results[f"pred_{ep}_std"] = blend_std
    train_mean_ep = np.nanmean(Y_train_full[:, t])
    train_std_ep = np.nanstd(Y_train_full[:, t])
    screen_results[f"{ep}_zscore_vs_train"] = (blend_mean - train_mean_ep) / (train_std_ep + 1e-9)

screen_results = pd.concat([screen_results, ad_scores.reset_index(drop=True)], axis=1)
screen_results.head()


## 15. Part E.4 — Uncertainty vs Applicability Domain

Compounds that are *both* far outside the training distribution (high AD distance)
*and* have high model uncertainty are the ones to trust least; compounds flagged OOD
but with low model uncertainty are worth a second look (the model is confident, but
extrapolating).


In [ ]:
fig, axes = plt.subplots(1, len(CONFIG.endpoint_names), figsize=(6 * len(CONFIG.endpoint_names), 5))
if len(CONFIG.endpoint_names) == 1:
    axes = [axes]
for ax, ep in zip(axes, CONFIG.endpoint_names):
    sc = ax.scatter(screen_results["tanimoto_dissimilarity"], screen_results[f"pred_{ep}_std"],
                     c=screen_results["n_flags_OOD"], cmap="viridis", alpha=0.8, s=40)
    ax.set_xlabel("Tanimoto dissimilarity to nearest training compound")
    ax.set_ylabel(f"Blended prediction std ({ep})")
    ax.set_title(f"{ep}: uncertainty vs. domain distance")
    plt.colorbar(sc, ax=ax, label="# AD flags tripped")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(data=ad_scores, x="n_flags_OOD", discrete=True, ax=axes[0])
axes[0].set_title("Screening library: how many AD metrics flag each compound as OOD")
axes[0].set_xlabel("# of 4 AD metrics tripped")

pca_train_2d = PCA(n_components=2, random_state=CONFIG.random_state).fit(ad._train_desc_scaled)
train_2d = pca_train_2d.transform(ad._train_desc_scaled)
screen_desc_scaled = ad.scaler.transform(np.nan_to_num(screen_desc))
screen_2d = pca_train_2d.transform(screen_desc_scaled)
axes[1].scatter(train_2d[:, 0], train_2d[:, 1], alpha=0.4, label="PSMA training set", s=20)
axes[1].scatter(screen_2d[:, 0], screen_2d[:, 1], alpha=0.6, label="Screening library", s=20,
                c=screen_results["consensus_OOD"].map({True: "red", False: "tab:green"}))
axes[1].set_title("Descriptor-space PCA: training domain vs. screening library\n(red = consensus OOD)")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].legend()
plt.tight_layout()
plt.show()


## 16. Part E.5 — Top hits

Ranked by predicted `pIC50` among compounds inside the applicability domain
(`consensus_OOD == False`), with `pKi` and both metrics' uncertainty shown alongside
for context.


In [ ]:
cols = (["compound_id"] if "compound_id" in screen_results.columns else []) + \
       [f"pred_{ep}" for ep in CONFIG.endpoint_names] + \
       [f"pred_{ep}_std" for ep in CONFIG.endpoint_names] + \
       ["n_flags_OOD", "consensus_OOD", "tanimoto_dissimilarity"]

in_domain = screen_results[~screen_results["consensus_OOD"]]
top_hits = in_domain.sort_values(f"pred_{CONFIG.endpoint_names[0]}", ascending=False)[cols].head(15)
print(f"{len(in_domain):,} / {len(screen_results):,} screening compounds are inside the applicability domain.")
top_hits


## 16b. Part E.6 — Which atoms drive the top predictions?

Following An, Sherman & Dixon's Canvas-KPLS interpretability method (*J. Chem. Inf.
Model.* 2013): for each "on" fingerprint bit in a molecule, perturb its value from
1 to 1.05, see how much the (blended NN + GBM + KPLS) prediction moves, and
distribute that sensitivity equally over the atoms in the substructure that bit
represents (RDKit's Morgan-fingerprint `bitInfo` gives the atom environment for
every bit). Summing each atom's contributions across every fragment that contains it
gives a per-atom "favorable/unfavorable" score — turning a black-box ensemble
prediction into something a medicinal chemist can act on directly, rather than just
a number to accept or reject.

Atoms colored **red** push the prediction up (favorable for potency); atoms colored
**blue** pull it down. This only applies to compounds inside the fingerprint part of
the feature space, so it's skipped if `CONFIG.feature_set` ended up `"descriptors_only"`.


In [ ]:
from rdkit.Chem import Draw
from collections import defaultdict


def get_bit_atoms(mol: Chem.Mol, bit_info: dict, bit_id: int) -> set:
    '''Maps a Morgan-fingerprint bit back to the set of atom indices in its environment.'''
    atoms = set()
    for atom_idx, radius in bit_info[bit_id]:
        if radius == 0:
            atoms.add(atom_idx)
        else:
            env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
            amap: dict = {}
            Chem.PathToSubmol(mol, env, atomMap=amap)
            atoms.update(amap.keys())
    return atoms


def predict_blend_single(x_row: np.ndarray, task_idx: int, ep: str) -> float:
    '''Point prediction from the deployed blended model (NN in eval/no-dropout mode +
    GBM + KPLS) for a single feature vector — used by the perturbation analysis below.'''
    x_batch = x_row.reshape(1, -1).astype(np.float32)
    with torch.no_grad():
        nn_pred = nn_model_deploy.to(DEVICE).eval().forward_tasks(
            torch.tensor(x_batch, dtype=torch.float32, device=DEVICE)
        ).cpu().numpy()[0, task_idx]
    gbm_pred = gbm_models_deploy[ep].predict(x_batch)[0]
    kpls_pred = kpls_predict(kpls_models_deploy[ep], x_batch)[0]
    w = blend_weights[ep]
    return w["nn"] * nn_pred + w["gbm"] * gbm_pred + w["kpls"] * kpls_pred


def compute_atom_sensitivities(mol: Chem.Mol, x_row: np.ndarray, task_idx: int, ep: str,
                                perturb: float = 0.05) -> Dict[int, float]:
    _, bit_info = featurizer.morgan_fp_with_bit_info(mol)
    base_pred = predict_blend_single(x_row, task_idx, ep)
    atom_scores: Dict[int, float] = defaultdict(float)
    on_bits = [b for b in bit_info if x_row[b] > 0.5]
    for b in on_bits:
        x_pert = x_row.copy()
        x_pert[b] = 1.0 + perturb
        sens = (predict_blend_single(x_pert, task_idx, ep) - base_pred) / perturb
        atoms = get_bit_atoms(mol, bit_info, b)
        if atoms:
            for a in atoms:
                atom_scores[a] += sens / len(atoms)
    return atom_scores


if CONFIG.feature_set == "descriptors_only":
    print("feature_set='descriptors_only' -> no fingerprint bits to attribute to atoms; "
          "skipping atom-level interpretability.")
else:
    top_n_interp = min(4, len(top_hits))
    fig, axes = plt.subplots(1, top_n_interp, figsize=(4.2 * top_n_interp, 4.2))
    if top_n_interp == 1:
        axes = [axes]
    cmap = plt.get_cmap("coolwarm")
    for ax, row_idx in zip(axes, top_hits.index[:top_n_interp]):
        mol = screen_feat["mols"][row_idx]
        x_row = X_screen[row_idx]
        atom_scores = compute_atom_sensitivities(mol, x_row, 0, CONFIG.endpoint_names[0])
        if not atom_scores:
            ax.set_title("no 'on' bits found"); ax.axis("off"); continue
        max_abs = max(abs(v) for v in atom_scores.values()) or 1.0
        atom_colors = {a: cmap(0.5 + 0.5 * (v / max_abs))[:3] for a, v in atom_scores.items()}
        img = Draw.MolToImage(mol, size=(320, 320), highlightAtoms=list(atom_scores.keys()),
                               highlightAtomColors=atom_colors)
        ax.imshow(img)
        ax.axis("off")
        cid = screen_results.loc[row_idx, "compound_id"] if "compound_id" in screen_results.columns else row_idx
        pred_val = screen_results.loc[row_idx, f"pred_{CONFIG.endpoint_names[0]}"]
        ax.set_title(f"{cid}\npred {CONFIG.endpoint_names[0]}={pred_val:.2f}", fontsize=9)
    plt.suptitle(f"Atom contributions to predicted {CONFIG.endpoint_names[0]} "
                 "(red = favorable, blue = unfavorable)", y=1.03)
    plt.tight_layout()
    plt.show()


## 17. Part F — Save outputs

Everything needed to reproduce or redeploy this run: the screening predictions, all
three final model families (NN state dict, tuned LightGBM, tuned KPLS — each with
their bootstrap ensembles where applicable), the shared feature scaler, the
descriptor selector, the AD object, the held-out uncertainty-calibration data, and
every hyperparameter-search result (for an audit trail of what
`GridSearchCV`/`BayesSearchCV`/`Optuna` actually tried and picked, each scored by the
AutoQSAR train/test-gap-aware metric).


In [ ]:
out_dir = Path(CONFIG.output_dir)

screen_results.to_csv(out_dir / "screening_predictions.csv", index=False)

torch.save({
    # moved to CPU so the saved file loads fine on a machine without a GPU
    "trunk_state_dict": {k: v.cpu() for k, v in nn_model_deploy.trunk.state_dict().items()},
    "task_heads_state_dict": {k: v.cpu() for k, v in nn_model_deploy.task_heads.state_dict().items()},
    "hidden_dims": nn_hidden_dims_final,
    "dropout": nn_best_params["dropout"],
    "endpoint_names": list(CONFIG.endpoint_names),
}, out_dir / "nn_transfer_model.pt")

for ep in CONFIG.endpoint_names:
    joblib.dump(gbm_models_deploy[ep], out_dir / f"lightgbm_{ep}.joblib")
    joblib.dump(gbm_bootstrap_ensembles[ep], out_dir / f"lightgbm_{ep}_bootstrap_ensemble.joblib")
    joblib.dump(kpls_models_deploy[ep], out_dir / f"kpls_{ep}.joblib")
    joblib.dump(kpls_bootstrap_ensembles[ep], out_dir / f"kpls_{ep}_bootstrap_ensemble.joblib")

joblib.dump(feature_scaler, out_dir / "feature_scaler.joblib")
joblib.dump(descriptor_selector, out_dir / "descriptor_selector.joblib")
joblib.dump(ad, out_dir / "applicability_domain.joblib")

tuning_summary = {
    "feature_set_comparison": _fs_scores,
    "selected_feature_set": CONFIG.feature_set,
    "n_descriptors_raw": len(descriptor_names_raw),
    "n_descriptors_curated": len(descriptor_names),
    "gbm_tuning_results": {ep: {k: {"autoqsar_score": v[0], "best_params": v[1], "train_R2": v[2], "test_R2": v[3]}
                                 for k, v in d.items()} for ep, d in gbm_tuning_results.items()},
    "kpls_tuning_results": {ep: {k: {"autoqsar_score": v[0], "best_params": v[1], "train_R2": v[2], "test_R2": v[3]}
                                  for k, v in d.items()} for ep, d in kpls_tuning_results.items()},
    "gbm_best_params": gbm_best_params,
    "kpls_best_params": kpls_best_params,
    "nn_best_params": nn_best_params,
    "blend_weights": blend_weights,
    "held_out_eval_single_split": eval_df.to_dict(orient="records"),
    "held_out_eval_repeated_mean_std": {
        f"{ep}|{model}": {"R2_mean": row[("R2", "mean")], "R2_std": row[("R2", "std")],
                           "RMSE_mean": row[("RMSE", "mean")], "RMSE_std": row[("RMSE", "std")]}
        for (ep, model), row in repeated_eval_summary.iterrows()
    },
}
with open(out_dir / "hyperparameter_tuning_summary.json", "w") as f:
    json.dump(tuning_summary, f, indent=2, default=str)

calibration_df.to_csv(out_dir / "held_out_uncertainty_calibration.csv", index=False)

with open(out_dir / "config.json", "w") as f:
    json.dump({k: (list(v) if isinstance(v, (tuple, list)) else v) for k, v in CONFIG.__dict__.items()}, f, indent=2)

print(f"Saved all outputs to {out_dir}/:")
for p in sorted(out_dir.glob("*")):
    print(" -", p.name)


## 18. Notes for reuse

- **New target instead of PSMA**: change `target_uniprot_id` / `target_name_like` /
  `organism_like` in `CONFIG` for the BindingDB fine-tuning set, **and** update
  `chembl_pretrain_target_keywords` to reflect *that* target's related family — the
  domain-targeted pretraining set is only as useful as the keyword list describing
  what "related" means for the new target.
- **Pretraining scope**: `chembl_pretrain_mode = "domain_targeted"` (default) restricts
  ChEMBL pretraining to targets matching `chembl_pretrain_target_keywords` — other
  zinc metallopeptidases for PSMA, by default. Set it to `"broad"` to go back to an
  arbitrary target-agnostic sample, or edit the keyword list to widen/narrow "related."
  If fewer than `chembl_pretrain_min_targets_or_fallback_to_broad` targets match, it
  automatically falls back to broad mode for that run (printed clearly either way).
- **Different affinity endpoints**: edit `affinity_columns` / `endpoint_names`
  together (they're matched by position) — any subset of BindingDB's affinity
  columns works, not just IC50/Ki, and the multi-task machinery (masked loss, per-task
  heads) generalizes to any number of tasks, not just two.
- **Faster iteration while developing**: drop `optuna_n_trials`, `bayes_search_n_iter`,
  `nn_pretrain_epochs`, and `nn_finetune_total_epochs`, and/or set
  `chembl_pretrain_max_compounds` lower — everything downstream reads from `CONFIG`.
- **No internet / dry run**: every network call (ChEMBL, BindingDB) has a
  synthetic-data fallback, so the whole notebook still runs (with obviously fake
  data, clearly printed) for a structural smoke test.
- **Re-tuning without re-pretraining**: `pretrained_final.trunk.state_dict()` is
  reusable across as many fine-tuning experiments as you like — cache it
  (`torch.save`) if you plan to try many fine-tuning configurations by hand outside
  Optuna's search.
